In [44]:
import os

import mlflow

from paretodo.dashboard.callbacks.pareto_front import (
    update_pareto_front_plots,
    update_pareto_front_parcoords,
)
from paretodo.dashboard.utils.types_and_constants import LOCAL_TEMP_DIR

from setup.plotly_theme import register_theme

register_theme()

PDF_FOLDER = ".local/pdf"
os.makedirs(PDF_FOLDER, exist_ok=True)


In [45]:
# MLflow tracking URI
# Local SQLite:  "sqlite:////absolute/path/to/mlflow.db"
# Running server: "http://127.0.0.1:5000"
MLFLOW_TRACKING_URI = "sqlite:////Users/ss15425/Documents/00_Git_Repositories/pfgs-research/mlflow.db"

MLFLOW_EXPERIMENT = "Journal-02-06-2026"
MLFLOW_RUN = "HIDL_8D"
ITERATION = "Iteration_001"

# Renames: map internal column names to display labels (leave empty for defaults)
OBJECTIVE_RENAMES: dict[str, str] = {}
INVESTIGATION_SPACE_RENAMES: dict[str, str] = {}
MEASUREMENT_RENAMES: dict[str, str] = {}

# Bounds — keys must match the exact objective column names printed by cell 3.
# Points outside ANY bound appear as black "Discarded" markers and red dashed lines.
# Uncomment and fill in the bounds you want:
OBJECTIVES_BOUNDS = {
    # "obj:max(μ(product_concentration))": {"min": 1000, "max": None},
    # "obj:max(R(product_concentration))": {"min": 1000,   "max": None},
    # "obj:max(P(purity))":                {"min": 0.6,  "max": None},
}
# OBJECTIVES_BOUNDS: dict[str, dict] = {}

PARETO_MODE = "Optimization"  # "Optimization" or "Manufacturing"

x_axis_names = ["μ(Titer) [mg/L]", "μ(Titer) [mg/L]", "μ(Titer) [mg/L]"]
y_axis_names = ["σ(Titer) [mg/L]", "Robustness(Titer) [mg/L]", "P(Purity) [-]"]
subplot_titles = ["Mean Prediction vs. Standard Deviation", "Mean Prediction vs. Robustness", "Mean Prediction vs. Constraint Probability"]


In [46]:
import pandas as pd

client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
runs = client.search_runs(experiment_ids=[experiment.experiment_id])
selected_run = next(r for r in runs if r.info.run_name == MLFLOW_RUN)
run_id = selected_run.info.run_id

temp_dir = os.path.join(LOCAL_TEMP_DIR, MLFLOW_RUN)
os.makedirs(temp_dir, exist_ok=True)

client.download_artifacts(run_id=run_id, path="general", dst_path=temp_dir)
client.download_artifacts(run_id=run_id, path=ITERATION, dst_path=temp_dir)

print(f"Artifacts downloaded to: {temp_dir}")

# Show objective column names — use these as keys in OBJECTIVES_BOUNDS
_pareto_csv = f"{temp_dir}/{ITERATION}/pareto_front_predictions.csv"
_obj_cols = [c for c in pd.read_csv(_pareto_csv, nrows=0).columns if c.startswith("obj:")]
print("\nObjective columns (use as OBJECTIVES_BOUNDS keys):")
for c in _obj_cols:
    print(f"  {c!r}")


2026-10-05 16:20:06,597 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:06,598 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.
2026-10-05 16:20:06,598 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:06,598 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.

Artifacts downloaded to: .temp/HIDL_8D

Objective columns (use as OBJECTIVES_BOUNDS keys):
  'obj:max(μ(product_concentration))'
  'obj:max(σ(product_concentration))'
  'obj:max(R(product_concentration))'
  'obj:max(P(purity))'


In [47]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_full = update_pareto_front_plots(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    clicked_points={},
    objectives_bounds=OBJECTIVES_BOUNDS,
    pinned_points={},
    objective_renames=OBJECTIVE_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    measurement_renames=MEASUREMENT_RENAMES,
    pareto_mode=PARETO_MODE,
)

# ── Row 1 only ────────────────────────────────────────────────────────────────
layout_dict = fig_full.to_plotly_json()["layout"]

# Row 1 yaxes: domain[1] ≈ 1.0 (topmost row always reaches the top)
row1_y_refs = {
    "y" + key[5:]
    for key, val in layout_dict.items()
    if re.match(r"^yaxis\d*$", key)
    and isinstance(val, dict)
    and abs((val.get("domain") or [0, 0])[1] - 1.0) < 0.01
}
row1_x_refs = {"x" + ref[1:] for ref in row1_y_refs}

# Map axis ref → subplot column (sorted: 'x'→1, 'x2'→2, 'x3'→3)
def _col_num(ref: str) -> int:
    n = ref[1:]  # strip leading 'x'
    return int(n) if n else 1

sorted_x_refs = sorted(row1_x_refs, key=_col_num)
x_ref_to_col = {ref: i + 1 for i, ref in enumerate(sorted_x_refs)}
n_cols = len(sorted_x_refs)

# Build proper 1-row subplot figure
fig = make_subplots(rows=1, cols=n_cols, horizontal_spacing=0.12, subplot_titles=subplot_titles[:n_cols])
fig.update_layout(template=fig_full.layout.template)

# Add row-1 traces with explicit col so xaxis/yaxis are always set correctly
for trace in fig_full.data:
    x_ref = trace.xaxis or "x"
    y_ref = trace.yaxis or "y"
    if y_ref in row1_y_refs:
        col = x_ref_to_col.get(x_ref, 1)
        fig.add_trace(trace, row=1, col=col)

# Copy axis titles/ticks from original (skip domain; make_subplots handles it)
for key, val in layout_dict.items():
    if not isinstance(val, dict):
        continue
    if re.match(r"^xaxis\d*$", key) and "x" + key[5:] in row1_x_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})
    elif re.match(r"^yaxis\d*$", key) and "y" + key[5:] in row1_y_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})

# Copy row-1 shapes (bounds vlines/hlines) using subplot-aware placement
for shape in fig_full.layout.shapes:
    s = shape.to_plotly_json()
    xref_base = (s.get("xref") or "").split(" ")[0]
    if xref_base not in row1_x_refs:
        continue
    col = x_ref_to_col.get(xref_base, 1)
    line = s.get("line", {})
    x0, x1 = s.get("x0"), s.get("x1")
    y0, y1 = s.get("y0"), s.get("y1")
    if x0 is not None and x0 == x1:          # vertical bound line
        fig.add_vline(x=float(x0), row=1, col=col, line=line)
    elif y0 is not None and y0 == y1:         # horizontal bound line
        fig.add_hline(y=float(y0), row=1, col=col, line=line)
    else:                                     # diagonal or other shape
        fig.add_shape(
            **{k: v for k, v in s.items() if k not in ("xref", "yref")},
            row=1, col=col,
        )

fig.update_layout(
    showlegend=layout_dict.get("showlegend", True),
    hovermode=layout_dict.get("hovermode"),
    legend=layout_dict.get("legend"),
    width=layout_dict.get("width"),
    height=500,
)

# ── Diagonal gray dashed line on second subplot (col 2) ───────────────────────
x2_vals, y2_vals = [], []
for trace in fig.data:
    if trace.xaxis == "x2" and trace.yaxis == "y2":
        if getattr(trace, "x", None) is not None:
            x2_vals.extend(v for v in trace.x if v is not None)
        if getattr(trace, "y", None) is not None:
            y2_vals.extend(v for v in trace.y if v is not None)

if x2_vals and y2_vals:
    diag_min = min(min(x2_vals), min(y2_vals))
    diag_max = max(max(x2_vals), max(y2_vals))
    fig.add_shape(
        type="line",
        x0=diag_min, y0=diag_min,
        x1=diag_max, y1=diag_max,
        row=1, col=2,
        line=dict(color="gray", width=2, dash="dash"),
    )

fig.update_layout(showlegend=False, margin=dict(l=240, r=20, t=110, b=120), width=2200, height=600,legend=dict(font=dict(size=30)))
for col_idx, (x_name, y_name) in enumerate(zip(x_axis_names, y_axis_names), start=1):
    fig.update_xaxes(title_text=x_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30),)
    fig.update_yaxes(title_text=y_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30))
for annotation in fig.layout.annotations:
    annotation.font.size = 25

# Iteration label on the left, spanning the whole row
fig.add_annotation(
    text=f"<b>Iteration {int(ITERATION.split('_')[-1])}</b>",
    x=-0.09, y=0.5,
    xref="paper", yref="paper",
    textangle=-90, xanchor="center", yanchor="middle",
    showarrow=False,
    font=dict(size=32),
)
fig.show()
fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.write_image(f"{PDF_FOLDER}/pareto_front_all.pdf")


2026-10-05 16:20:06,711 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:06,712 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:06,713 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp6q_wdk_3.
2026-10-05 16:20:06,714 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:06,714 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp0yrlsxgu.
2026-10-05 16:20:06,715 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp0yrlsxgu
2026-10-05 16:20:07,513 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp6q_wdk_3/index.html
2026-10-05 16:20:08,131 - kaleido.kaleido - INFO - Getting tab from queue 

In [48]:
# MLflow tracking URI
# Local SQLite:  "sqlite:////absolute/path/to/mlflow.db"
# Running server: "http://127.0.0.1:5000"
MLFLOW_TRACKING_URI = "sqlite:////Users/ss15425/Documents/00_Git_Repositories/pfgs-research/mlflow.db"

MLFLOW_EXPERIMENT = "Journal-02-06-2026"
MLFLOW_RUN = "HIDL_8D"
ITERATION = "Iteration_001"

# Renames: map internal column names to display labels (leave empty for defaults)
OBJECTIVE_RENAMES: dict[str, str] = {}
INVESTIGATION_SPACE_RENAMES: dict[str, str] = {}
MEASUREMENT_RENAMES: dict[str, str] = {}

# Bounds — keys must match the exact objective column names printed by cell 3.
# Points outside ANY bound appear as black "Discarded" markers and red dashed lines.
# Uncomment and fill in the bounds you want:
OBJECTIVES_BOUNDS = {
    "obj:max(μ(product_concentration))": {"min": 1000, "max": None},
    # "obj:max(R(product_concentration))": {"min": 1000,   "max": None},
    # "obj:max(P(purity))":                {"min": 0.6,  "max": None},
}
# OBJECTIVES_BOUNDS: dict[str, dict] = {}

PARETO_MODE = "Optimization"  # "Optimization" or "Manufacturing"

x_axis_names = ["μ(Titer) [mg/L]", "μ(Titer) [mg/L]", "μ(Titer) [mg/L]"]
y_axis_names = ["σ(Titer) [mg/L]", "Robustness(Titer) [mg/L]", "P(Purity) [-]"]
subplot_titles = ["Mean Prediction vs. Standard Deviation", "Mean Prediction vs. Robustness", "Mean Prediction vs. Constraint Probability"]


In [49]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_full = update_pareto_front_plots(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    clicked_points={},
    objectives_bounds=OBJECTIVES_BOUNDS,
    pinned_points={},
    objective_renames=OBJECTIVE_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    measurement_renames=MEASUREMENT_RENAMES,
    pareto_mode=PARETO_MODE,
)

# ── Row 1 only ────────────────────────────────────────────────────────────────
layout_dict = fig_full.to_plotly_json()["layout"]

# Row 1 yaxes: domain[1] ≈ 1.0 (topmost row always reaches the top)
row1_y_refs = {
    "y" + key[5:]
    for key, val in layout_dict.items()
    if re.match(r"^yaxis\d*$", key)
    and isinstance(val, dict)
    and abs((val.get("domain") or [0, 0])[1] - 1.0) < 0.01
}
row1_x_refs = {"x" + ref[1:] for ref in row1_y_refs}

# Map axis ref → subplot column (sorted: 'x'→1, 'x2'→2, 'x3'→3)
def _col_num(ref: str) -> int:
    n = ref[1:]  # strip leading 'x'
    return int(n) if n else 1

sorted_x_refs = sorted(row1_x_refs, key=_col_num)
x_ref_to_col = {ref: i + 1 for i, ref in enumerate(sorted_x_refs)}
n_cols = len(sorted_x_refs)

# Build proper 1-row subplot figure
fig = make_subplots(rows=1, cols=n_cols, horizontal_spacing=0.12, subplot_titles=subplot_titles[:n_cols])
fig.update_layout(template=fig_full.layout.template)

# Add row-1 traces with explicit col so xaxis/yaxis are always set correctly
for trace in fig_full.data:
    x_ref = trace.xaxis or "x"
    y_ref = trace.yaxis or "y"
    if y_ref in row1_y_refs:
        col = x_ref_to_col.get(x_ref, 1)
        fig.add_trace(trace, row=1, col=col)

# Copy axis titles/ticks from original (skip domain; make_subplots handles it)
for key, val in layout_dict.items():
    if not isinstance(val, dict):
        continue
    if re.match(r"^xaxis\d*$", key) and "x" + key[5:] in row1_x_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})
    elif re.match(r"^yaxis\d*$", key) and "y" + key[5:] in row1_y_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})

# Copy row-1 shapes (bounds vlines/hlines) using subplot-aware placement
for shape in fig_full.layout.shapes:
    s = shape.to_plotly_json()
    xref_base = (s.get("xref") or "").split(" ")[0]
    if xref_base not in row1_x_refs:
        continue
    col = x_ref_to_col.get(xref_base, 1)
    line = s.get("line", {})
    x0, x1 = s.get("x0"), s.get("x1")
    y0, y1 = s.get("y0"), s.get("y1")
    if x0 is not None and x0 == x1:          # vertical bound line
        fig.add_vline(x=float(x0), row=1, col=col, line=line)
    elif y0 is not None and y0 == y1:         # horizontal bound line
        fig.add_hline(y=float(y0), row=1, col=col, line=line)
    else:                                     # diagonal or other shape
        fig.add_shape(
            **{k: v for k, v in s.items() if k not in ("xref", "yref")},
            row=1, col=col,
        )

fig.update_layout(
    showlegend=layout_dict.get("showlegend", True),
    hovermode=layout_dict.get("hovermode"),
    legend=layout_dict.get("legend"),
    width=layout_dict.get("width"),
    height=500,
)

# ── Diagonal gray dashed line on second subplot (col 2) ───────────────────────
x2_vals, y2_vals = [], []
for trace in fig.data:
    if trace.xaxis == "x2" and trace.yaxis == "y2":
        if getattr(trace, "x", None) is not None:
            x2_vals.extend(v for v in trace.x if v is not None)
        if getattr(trace, "y", None) is not None:
            y2_vals.extend(v for v in trace.y if v is not None)

if x2_vals and y2_vals:
    diag_min = min(min(x2_vals), min(y2_vals))
    diag_max = max(max(x2_vals), max(y2_vals))
    fig.add_shape(
        type="line",
        x0=diag_min, y0=diag_min,
        x1=diag_max, y1=diag_max,
        row=1, col=2,
        line=dict(color="gray", width=2, dash="dash"),
    )

fig.update_layout(showlegend=False, margin=dict(l=240, r=20, t=110, b=120), width=2200, height=600,legend=dict(font=dict(size=30)))
for col_idx, (x_name, y_name) in enumerate(zip(x_axis_names, y_axis_names), start=1):
    fig.update_xaxes(title_text=x_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30),)
    fig.update_yaxes(title_text=y_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30))
for annotation in fig.layout.annotations:
    annotation.font.size = 25

# Iteration label on the left, spanning the whole row
fig.add_annotation(
    text=f"<b>Iteration {int(ITERATION.split('_')[-1])}</b>",
    x=-0.09, y=0.5,
    xref="paper", yref="paper",
    textangle=-90, xanchor="center", yanchor="middle",
    showarrow=False,
    font=dict(size=32),
)
fig.show()
fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.write_image(f"{PDF_FOLDER}/pareto_front_mean.pdf")

2026-10-05 16:20:08,516 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:08,516 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-10-05 16:20:08,604 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:08,605 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:08,606 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpc0dvqfvy.
2026-10-05 16:20:08,606 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:08,607 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpmth5ymu_.
2026-10-05 16:20:08,607 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpmth5ymu_
2026-10-05 16:20:09,024 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpc0dvqfvy/index.html
2026-10-05 16:20:09,587 - kaleido.kaleido - INFO - Getting tab from queue 

In [50]:
# MLflow tracking URI
# Local SQLite:  "sqlite:////absolute/path/to/mlflow.db"
# Running server: "http://127.0.0.1:5000"
MLFLOW_TRACKING_URI = "sqlite:////Users/ss15425/Documents/00_Git_Repositories/pfgs-research/mlflow.db"

MLFLOW_EXPERIMENT = "Journal-02-06-2026"
MLFLOW_RUN = "HIDL_8D"
ITERATION = "Iteration_001"

# Renames: map internal column names to display labels (leave empty for defaults)
OBJECTIVE_RENAMES: dict[str, str] = {}
INVESTIGATION_SPACE_RENAMES: dict[str, str] = {}
MEASUREMENT_RENAMES: dict[str, str] = {}

# Bounds — keys must match the exact objective column names printed by cell 3.
# Points outside ANY bound appear as black "Discarded" markers and red dashed lines.
# Uncomment and fill in the bounds you want:
OBJECTIVES_BOUNDS = {
    "obj:max(μ(product_concentration))": {"min": 1000, "max": None},
    # "obj:max(R(product_concentration))": {"min": 1000,   "max": None},
    "obj:max(P(purity))":                {"min": 0.6,  "max": None},
}
# OBJECTIVES_BOUNDS: dict[str, dict] = {}

PARETO_MODE = "Optimization"  # "Optimization" or "Manufacturing"

x_axis_names = ["μ(Titer) [mg/L]", "μ(Titer) [mg/L]", "μ(Titer) [mg/L]"]
y_axis_names = ["σ(Titer) [mg/L]", "Robustness(Titer) [mg/L]", "P(Purity) [-]"]
subplot_titles = ["Mean Prediction vs. Standard Deviation", "Mean Prediction vs. Robustness", "Mean Prediction vs. Constraint Probability"]


In [51]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_full = update_pareto_front_plots(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    clicked_points={},
    objectives_bounds=OBJECTIVES_BOUNDS,
    pinned_points={},
    objective_renames=OBJECTIVE_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    measurement_renames=MEASUREMENT_RENAMES,
    pareto_mode=PARETO_MODE,
)

# ── Row 1 only ────────────────────────────────────────────────────────────────
layout_dict = fig_full.to_plotly_json()["layout"]

# Row 1 yaxes: domain[1] ≈ 1.0 (topmost row always reaches the top)
row1_y_refs = {
    "y" + key[5:]
    for key, val in layout_dict.items()
    if re.match(r"^yaxis\d*$", key)
    and isinstance(val, dict)
    and abs((val.get("domain") or [0, 0])[1] - 1.0) < 0.01
}
row1_x_refs = {"x" + ref[1:] for ref in row1_y_refs}

# Map axis ref → subplot column (sorted: 'x'→1, 'x2'→2, 'x3'→3)
def _col_num(ref: str) -> int:
    n = ref[1:]  # strip leading 'x'
    return int(n) if n else 1

sorted_x_refs = sorted(row1_x_refs, key=_col_num)
x_ref_to_col = {ref: i + 1 for i, ref in enumerate(sorted_x_refs)}
n_cols = len(sorted_x_refs)

# Build proper 1-row subplot figure
fig = make_subplots(rows=1, cols=n_cols, horizontal_spacing=0.12, subplot_titles=subplot_titles[:n_cols])
fig.update_layout(template=fig_full.layout.template)

# Add row-1 traces with explicit col so xaxis/yaxis are always set correctly
for trace in fig_full.data:
    x_ref = trace.xaxis or "x"
    y_ref = trace.yaxis or "y"
    if y_ref in row1_y_refs:
        col = x_ref_to_col.get(x_ref, 1)
        fig.add_trace(trace, row=1, col=col)

# Copy axis titles/ticks from original (skip domain; make_subplots handles it)
for key, val in layout_dict.items():
    if not isinstance(val, dict):
        continue
    if re.match(r"^xaxis\d*$", key) and "x" + key[5:] in row1_x_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})
    elif re.match(r"^yaxis\d*$", key) and "y" + key[5:] in row1_y_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})

# Copy row-1 shapes (bounds vlines/hlines) using subplot-aware placement
for shape in fig_full.layout.shapes:
    s = shape.to_plotly_json()
    xref_base = (s.get("xref") or "").split(" ")[0]
    if xref_base not in row1_x_refs:
        continue
    col = x_ref_to_col.get(xref_base, 1)
    line = s.get("line", {})
    x0, x1 = s.get("x0"), s.get("x1")
    y0, y1 = s.get("y0"), s.get("y1")
    if x0 is not None and x0 == x1:          # vertical bound line
        fig.add_vline(x=float(x0), row=1, col=col, line=line)
    elif y0 is not None and y0 == y1:         # horizontal bound line
        fig.add_hline(y=float(y0), row=1, col=col, line=line)
    else:                                     # diagonal or other shape
        fig.add_shape(
            **{k: v for k, v in s.items() if k not in ("xref", "yref")},
            row=1, col=col,
        )

fig.update_layout(
    showlegend=layout_dict.get("showlegend", True),
    hovermode=layout_dict.get("hovermode"),
    legend=layout_dict.get("legend"),
    width=layout_dict.get("width"),
    height=500,
)

# ── Diagonal gray dashed line on second subplot (col 2) ───────────────────────
x2_vals, y2_vals = [], []
for trace in fig.data:
    if trace.xaxis == "x2" and trace.yaxis == "y2":
        if getattr(trace, "x", None) is not None:
            x2_vals.extend(v for v in trace.x if v is not None)
        if getattr(trace, "y", None) is not None:
            y2_vals.extend(v for v in trace.y if v is not None)

if x2_vals and y2_vals:
    diag_min = min(min(x2_vals), min(y2_vals))
    diag_max = max(max(x2_vals), max(y2_vals))
    fig.add_shape(
        type="line",
        x0=diag_min, y0=diag_min,
        x1=diag_max, y1=diag_max,
        row=1, col=2,
        line=dict(color="gray", width=2, dash="dash"),
    )

fig.update_layout(showlegend=False, margin=dict(l=240, r=20, t=110, b=120), width=2200, height=600,legend=dict(font=dict(size=30)))
for col_idx, (x_name, y_name) in enumerate(zip(x_axis_names, y_axis_names), start=1):
    fig.update_xaxes(title_text=x_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30),)
    fig.update_yaxes(title_text=y_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30))
for annotation in fig.layout.annotations:
    annotation.font.size = 25

# Iteration label on the left, spanning the whole row
fig.add_annotation(
    text=f"<b>Iteration {int(ITERATION.split('_')[-1])}</b>",
    x=-0.09, y=0.5,
    xref="paper", yref="paper",
    textangle=-90, xanchor="center", yanchor="middle",
    showarrow=False,
    font=dict(size=32),
)
fig.show()
fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.write_image(f"{PDF_FOLDER}/pareto_front_p.pdf")

2026-10-05 16:20:09,947 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:09,948 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-10-05 16:20:10,039 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:10,039 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:10,040 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp0otlw784.
2026-10-05 16:20:10,041 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:10,041 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpmw5_nyhi.
2026-10-05 16:20:10,042 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpmw5_nyhi
2026-10-05 16:20:10,458 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp0otlw784/index.html
2026-10-05 16:20:11,190 - kaleido.kaleido - INFO - Getting tab from queue 

In [52]:
# MLflow tracking URI
# Local SQLite:  "sqlite:////absolute/path/to/mlflow.db"
# Running server: "http://127.0.0.1:5000"
MLFLOW_TRACKING_URI = "sqlite:////Users/ss15425/Documents/00_Git_Repositories/pfgs-research/mlflow.db"

MLFLOW_EXPERIMENT = "Journal-02-06-2026"
MLFLOW_RUN = "HIDL_8D"
ITERATION = "Iteration_001"

# Renames: map internal column names to display labels (leave empty for defaults)
OBJECTIVE_RENAMES: dict[str, str] = {}
INVESTIGATION_SPACE_RENAMES: dict[str, str] = {}
MEASUREMENT_RENAMES: dict[str, str] = {}

# Bounds — keys must match the exact objective column names printed by cell 3.
# Points outside ANY bound appear as black "Discarded" markers and red dashed lines.
# Uncomment and fill in the bounds you want:
OBJECTIVES_BOUNDS = {
    "obj:max(μ(product_concentration))": {"min": 1000, "max": None},
    "obj:max(R(product_concentration))": {"min": 1000,   "max": None},
    "obj:max(P(purity))":                {"min": 0.6,  "max": None},
}
# OBJECTIVES_BOUNDS: dict[str, dict] = {}

PARETO_MODE = "Optimization"  # "Optimization" or "Manufacturing"

x_axis_names = ["μ(Titer) [mg/L]", "μ(Titer) [mg/L]", "μ(Titer) [mg/L]"]
y_axis_names = ["σ(Titer) [mg/L]", "Robustness(Titer) [mg/L]", "P(Purity) [-]"]
subplot_titles = ["Mean Prediction vs. Standard Deviation", "Mean Prediction vs. Robustness", "Mean Prediction vs. Constraint Probability"]


In [53]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_full = update_pareto_front_plots(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    clicked_points={},
    objectives_bounds=OBJECTIVES_BOUNDS,
    pinned_points={},
    objective_renames=OBJECTIVE_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    measurement_renames=MEASUREMENT_RENAMES,
    pareto_mode=PARETO_MODE,
)

# ── Row 1 only ────────────────────────────────────────────────────────────────
layout_dict = fig_full.to_plotly_json()["layout"]

# Row 1 yaxes: domain[1] ≈ 1.0 (topmost row always reaches the top)
row1_y_refs = {
    "y" + key[5:]
    for key, val in layout_dict.items()
    if re.match(r"^yaxis\d*$", key)
    and isinstance(val, dict)
    and abs((val.get("domain") or [0, 0])[1] - 1.0) < 0.01
}
row1_x_refs = {"x" + ref[1:] for ref in row1_y_refs}

# Map axis ref → subplot column (sorted: 'x'→1, 'x2'→2, 'x3'→3)
def _col_num(ref: str) -> int:
    n = ref[1:]  # strip leading 'x'
    return int(n) if n else 1

sorted_x_refs = sorted(row1_x_refs, key=_col_num)
x_ref_to_col = {ref: i + 1 for i, ref in enumerate(sorted_x_refs)}
n_cols = len(sorted_x_refs)

# Build proper 1-row subplot figure
fig = make_subplots(rows=1, cols=n_cols, horizontal_spacing=0.12, subplot_titles=subplot_titles[:n_cols])
fig.update_layout(template=fig_full.layout.template)

# Add row-1 traces with explicit col so xaxis/yaxis are always set correctly
for trace in fig_full.data:
    x_ref = trace.xaxis or "x"
    y_ref = trace.yaxis or "y"
    if y_ref in row1_y_refs:
        col = x_ref_to_col.get(x_ref, 1)
        fig.add_trace(trace, row=1, col=col)

# Copy axis titles/ticks from original (skip domain; make_subplots handles it)
for key, val in layout_dict.items():
    if not isinstance(val, dict):
        continue
    if re.match(r"^xaxis\d*$", key) and "x" + key[5:] in row1_x_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})
    elif re.match(r"^yaxis\d*$", key) and "y" + key[5:] in row1_y_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})

# Copy row-1 shapes (bounds vlines/hlines) using subplot-aware placement
for shape in fig_full.layout.shapes:
    s = shape.to_plotly_json()
    xref_base = (s.get("xref") or "").split(" ")[0]
    if xref_base not in row1_x_refs:
        continue
    col = x_ref_to_col.get(xref_base, 1)
    line = s.get("line", {})
    x0, x1 = s.get("x0"), s.get("x1")
    y0, y1 = s.get("y0"), s.get("y1")
    if x0 is not None and x0 == x1:          # vertical bound line
        fig.add_vline(x=float(x0), row=1, col=col, line=line)
    elif y0 is not None and y0 == y1:         # horizontal bound line
        fig.add_hline(y=float(y0), row=1, col=col, line=line)
    else:                                     # diagonal or other shape
        fig.add_shape(
            **{k: v for k, v in s.items() if k not in ("xref", "yref")},
            row=1, col=col,
        )

fig.update_layout(
    showlegend=layout_dict.get("showlegend", True),
    hovermode=layout_dict.get("hovermode"),
    legend=layout_dict.get("legend"),
    width=layout_dict.get("width"),
    height=500,
)

# ── Diagonal gray dashed line on second subplot (col 2) ───────────────────────
x2_vals, y2_vals = [], []
for trace in fig.data:
    if trace.xaxis == "x2" and trace.yaxis == "y2":
        if getattr(trace, "x", None) is not None:
            x2_vals.extend(v for v in trace.x if v is not None)
        if getattr(trace, "y", None) is not None:
            y2_vals.extend(v for v in trace.y if v is not None)

if x2_vals and y2_vals:
    diag_min = min(min(x2_vals), min(y2_vals))
    diag_max = max(max(x2_vals), max(y2_vals))
    fig.add_shape(
        type="line",
        x0=diag_min, y0=diag_min,
        x1=diag_max, y1=diag_max,
        row=1, col=2,
        line=dict(color="gray", width=2, dash="dash"),
    )

fig.update_layout(showlegend=False, margin=dict(l=240, r=20, t=110, b=120), width=2200, height=600,legend=dict(font=dict(size=30)))
for col_idx, (x_name, y_name) in enumerate(zip(x_axis_names, y_axis_names), start=1):
    fig.update_xaxes(title_text=x_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30),)
    fig.update_yaxes(title_text=y_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30))
for annotation in fig.layout.annotations:
    annotation.font.size = 25

# Iteration label on the left, spanning the whole row
fig.add_annotation(
    text=f"<b>Iteration {int(ITERATION.split('_')[-1])}</b>",
    x=-0.09, y=0.5,
    xref="paper", yref="paper",
    textangle=-90, xanchor="center", yanchor="middle",
    showarrow=False,
    font=dict(size=32),
)
fig.show()


2026-10-05 16:20:11,501 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:11,501 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


In [54]:
import pandas as pd

# ── Recipe selection (0-based row indices in the Pareto front CSV) ─────────────
EXPERIMENT_RECIPES = [34,176,184,82,27]   # shown as orange diamonds,  e.g. [0, 5, 12]
HIGHLIGHT_RECIPES  = []   # shown as coloured X marks, e.g. [3, 7]

# ── Colour palette for highlight X marks (one colour per recipe) ──────────────
HIGHLIGHT_COLORS = [
    "#E72031", "#108173", "#E9AC11", "#EC720E",
    "#05374B", "#7AF6FA", "#003A91", "#030912",
]

# ─────────────────────────────────────────────────────────────────────────────
pareto_df = pd.read_csv(
    f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/{ITERATION}/pareto_front_predictions.csv"
)
obj_cols = [c for c in pareto_df.columns if c.startswith("obj:")]

num_objectives = len(obj_cols)
all_pairs = [(i, j) for i in range(num_objectives) for j in range(i + 1, num_objectives)]
n_cols = min(3, len(all_pairs))
row1_pairs = all_pairs[:n_cols]
axis_refs = [("x", "y"), ("x2", "y2"), ("x3", "y3")][:n_cols]

for subplot_idx, ((i, j), (xref, yref)) in enumerate(zip(row1_pairs, axis_refs)):
    x_col, y_col = obj_cols[i], obj_cols[j]

    # Orange diamonds
    if EXPERIMENT_RECIPES:
        sel = pareto_df.iloc[EXPERIMENT_RECIPES]
        fig.add_trace(go.Scatter(
            x=sel[x_col], y=sel[y_col],
            mode="markers",
            marker=dict(
                size=13,
                symbol="diamond",
                color="#FFA500",
                line=dict(width=2, color="black"),
            ),
            name="Experiments",
            legendgroup="experiments",
            showlegend=(subplot_idx == 0),
            xaxis=xref, yaxis=yref,
        ))

    # Coloured X marks — one trace per recipe so each gets its own legend entry
    for color_idx, rec_idx in enumerate(HIGHLIGHT_RECIPES):
        row = pareto_df.iloc[rec_idx]
        fig.add_trace(go.Scatter(
            x=[row[x_col]], y=[row[y_col]],
            mode="markers",
            marker=dict(
                size=14,
                symbol="x",
                color=HIGHLIGHT_COLORS[color_idx % len(HIGHLIGHT_COLORS)],
                line=dict(width=3),
            ),
            name=f"Recipe #{rec_idx}",
            legendgroup=f"highlight_{rec_idx}",
            showlegend=(subplot_idx == 0),
            xaxis=xref, yaxis=yref,
        ))
fig.update_layout(showlegend=True, legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.375), margin=dict(l=200, r=20, t=50, b=120), width=2320, height=600)
fig.show()

# Iteration label on the left, spanning the whole row
# fig.add_annotation(
#     text=f"<b>Iteration {int(ITERATION.split('_')[-1])}</b>",
#     x=-0.01, y=0.5,
#     xref="paper", yref="paper",
#     textangle=-90, xanchor="center", yanchor="middle",
#     showarrow=False,
#     font=dict(size=32),
# )

fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.write_image(f"{PDF_FOLDER}/results_{ITERATION}.pdf")


2026-10-05 16:20:11,611 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:11,612 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:11,612 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp4wj3y46x.
2026-10-05 16:20:11,613 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:11,614 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp7fh0xeeg.
2026-10-05 16:20:11,614 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp7fh0xeeg
2026-10-05 16:20:12,036 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp4wj3y46x/index.html
2026-10-05 16:20:12,698 - kaleido.kaleido - INFO - Getting tab from queue 

In [55]:
# Parallel coordinate plot: objectives + design space variables only
# Interactive: drag along any axis to select a range (click the axis again to clear)
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from paretodo.data_models import InvestigationSpace

PARETO_LINE_COLOR = "#C9D6FF"  # pale so the orange selected recipes stand out
SELECTED_LINE_COLOR = "#FF8C00"

PARCOORDS_LABELS = {
    "temperature": "Temperature [°C]",
    "ph_value": "pH [-]",
    "initial_glc": "Initial Glc",
    "initial_gln": "Initial Gln",
    "feed_1_glc": "Feed 1 Glc",
    "feed_1_gln": "Feed 1 Gln",
    "feed_2_glc": "Feed 2 Glc",
    "feed_2_gln": "Feed 2 Gln",
    "obj:max(μ(product_concentration))": "μ(Titer) [mg/L]",
    "obj:max(σ(product_concentration))": "σ(Titer) [mg/L]",
    "obj:max(R(product_concentration))": "Robustness(Titer) [mg/L]",
    "obj:max(P(purity))": "P(Purity) [-]",
}

_pareto_df = pd.read_csv(f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/{ITERATION}/pareto_front_predictions.csv")
_inv_space = InvestigationSpace.from_json(
    Path(f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/general/investigation_space.json")
)

# Selected recipes last so their lines are drawn on top
_selected = [i for i in EXPERIMENT_RECIPES if i < len(_pareto_df)]
_order = [i for i in range(len(_pareto_df)) if i not in _selected] + _selected
_plot_df = _pareto_df.iloc[_order]
_is_selected = [1 if i in _selected else 0 for i in _order]

# Design variables use the investigation-space bounds, objectives the data range
dims = [(v.name, v.min, v.max) for v in _inv_space.variables if v.name in _plot_df.columns]
dims += [(c, _plot_df[c].min(), _plot_df[c].max()) for c in _plot_df.columns if c.startswith("obj:")]

parcoords_fig = go.Figure(go.Parcoords(
    dimensions=[
        dict(label=PARCOORDS_LABELS.get(col, col), values=_plot_df[col], range=[float(lo), float(hi)])
        for col, lo, hi in dims
    ],
    line=dict(
        color=_is_selected, cmin=0, cmax=1, showscale=False,
        colorscale=[[0.0, PARETO_LINE_COLOR], [0.5, PARETO_LINE_COLOR],
                    [0.5, SELECTED_LINE_COLOR], [1.0, SELECTED_LINE_COLOR]],
    ),
    unselected=dict(line=dict(color="lightgray", opacity=0.2)),
    labelangle=-30,
    labelside="top",
    labelfont=dict(size=20),
    tickfont=dict(size=16),
    rangefont=dict(size=14),
))
parcoords_fig.update_layout(
    template=fig_full.layout.template,
    width=1500, height=800,
    title=dict(text="Pareto Front: Design Space and Objectives (orange = selected recipes)",
               font=dict(size=26), x=0.5, xanchor="center", y=0.98, yanchor="top"),
    margin=dict(l=120, r=120, t=260, b=60),
)
parcoords_fig.show()
parcoords_fig.write_image(f"{PDF_FOLDER}/pareto_parcoords.pdf")


2026-10-05 16:20:13,055 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:13,056 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:13,057 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.
2026-10-05 16:20:13,058 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:13,058 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpqlkyql2s.
2026-10-05 16:20:13,059 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:13,060 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpxk196b6o.
2026-10-05 16:20:13,060 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpxk196b6o
2026-10-05 16:20:13,475 - kaleido

In [56]:
# Static version of the parallel coordinate plot above with thicker lines (for the paper PDF)
# Reuses dims, _plot_df, _is_selected, PARCOORDS_LABELS and the line colours from the cell above
import numpy as np
import plotly.graph_objects as go

LINE_WIDTH = 2
SELECTED_LINE_WIDTH = 4
AXIS_COLOR = "#A0A0A0"
TEXT_COLOR = "#2a3f5f"
TICK_FONT_SIZE = 16
RANGE_FONT_SIZE = 14
LABEL_FONT_SIZE = 20


def _nice_ticks(lo, hi, n=10):
    raw = (hi - lo) / n
    mag = 10 ** np.floor(np.log10(raw))
    step = min((m * mag for m in (1, 2, 5, 10) if m * mag >= raw), default=10 * mag)
    return np.arange(np.ceil(lo / step) * step, hi + step * 1e-9, step)


def _fmt(v):
    return f"{v:.10g}"


def _range_fmt(v):
    return f"{v:,.1f}" if abs(v) >= 1000 else f"{v:.4g}"


n_dims = len(dims)
norm = np.column_stack([
    (_plot_df[col].to_numpy(dtype=float) - lo) / ((hi - lo) or 1) for col, lo, hi in dims
])  # shape (n_points, n_dims)
_sel_mask = np.array(_is_selected, dtype=bool)


def _lines_trace(rows, color, width, opacity=1.0):
    xs, ys = [], []
    for row in rows:
        xs += list(range(n_dims)) + [None]
        ys += row.tolist() + [None]
    return go.Scatter(x=xs, y=ys, mode="lines", line=dict(width=width, color=color), opacity=opacity,
                      showlegend=False, hoverinfo="skip")


parcoords_static_fig = go.Figure()
parcoords_static_fig.add_trace(_lines_trace(norm[~_sel_mask], PARETO_LINE_COLOR, LINE_WIDTH, opacity=0.6))
parcoords_static_fig.add_trace(_lines_trace(norm[_sel_mask], SELECTED_LINE_COLOR, SELECTED_LINE_WIDTH))

for k, (col, lo, hi) in enumerate(dims):
    # Axis line
    parcoords_static_fig.add_shape(type="line", x0=k, x1=k, y0=0, y1=1,
                                   line=dict(color=AXIS_COLOR, width=1), layer="above")
    # Ticks + tick labels (left of axis, white background like go.Parcoords)
    for t in _nice_ticks(lo, hi):
        y = (t - lo) / ((hi - lo) or 1)
        parcoords_static_fig.add_shape(type="line", x0=k - 0.03, x1=k, y0=y, y1=y,
                                       line=dict(color=AXIS_COLOR, width=1), layer="above")
        parcoords_static_fig.add_annotation(
            x=k, y=y, text=_fmt(t), showarrow=False, xanchor="right", xshift=-6,
            font=dict(size=TICK_FONT_SIZE, color=TEXT_COLOR), bgcolor="rgba(255,255,255,0.8)", borderpad=0,
        )
    # Range labels (min below, max above)
    parcoords_static_fig.add_annotation(x=k, y=1, text=_range_fmt(hi), showarrow=False, yanchor="bottom",
                                        yshift=4, font=dict(size=RANGE_FONT_SIZE, color=TEXT_COLOR))
    parcoords_static_fig.add_annotation(x=k, y=0, text=_range_fmt(lo), showarrow=False, yanchor="top",
                                        yshift=-4, font=dict(size=RANGE_FONT_SIZE, color=TEXT_COLOR))
    # Axis label (above, angled)
    parcoords_static_fig.add_annotation(
        x=k, y=1, text=PARCOORDS_LABELS.get(col, col), showarrow=False, textangle=-30,
        xanchor="left", yanchor="bottom", yshift=28, xshift=-10,
        font=dict(size=LABEL_FONT_SIZE, color=TEXT_COLOR),
    )

parcoords_static_fig.update_xaxes(visible=False, range=[-0.35, n_dims - 0.65])
parcoords_static_fig.update_yaxes(visible=False, range=[-0.02, 1.02])
parcoords_static_fig.update_layout(
    template=fig_full.layout.template,
    plot_bgcolor="white",
    width=1500, height=800,
    title=dict(text="Pareto Front: Design Space and Objectives (orange = selected recipes)",
               font=dict(size=26), x=0.5, xanchor="center", y=0.98, yanchor="top"),
    margin=dict(l=120, r=120, t=260, b=60),
)
parcoords_static_fig.show()
parcoords_static_fig.write_image(f"{PDF_FOLDER}/pareto_parcoords_thick.pdf")


2026-10-05 16:20:18,132 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:18,133 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:18,134 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpc55w7y5a.
2026-10-05 16:20:18,134 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:18,135 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpb7g5zjng.
2026-10-05 16:20:18,136 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpb7g5zjng
2026-10-05 16:20:18,562 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpc55w7y5a/index.html
2026-10-05 16:20:19,138 - kaleido.kaleido - INFO - Getting tab from queue 

# Iteration 2

In [57]:
# MLflow tracking URI
# Local SQLite:  "sqlite:////absolute/path/to/mlflow.db"
# Running server: "http://127.0.0.1:5000"
MLFLOW_TRACKING_URI = "sqlite:////Users/ss15425/Documents/00_Git_Repositories/pfgs-research/mlflow.db"

MLFLOW_EXPERIMENT = "Journal-02-06-2026"
MLFLOW_RUN = "HIDL_8D"
ITERATION = "Iteration_002"

# Renames: map internal column names to display labels (leave empty for defaults)
OBJECTIVE_RENAMES: dict[str, str] = {}
INVESTIGATION_SPACE_RENAMES: dict[str, str] = {}
MEASUREMENT_RENAMES: dict[str, str] = {}

# Bounds — keys must match the exact objective column names printed by cell 3.
# Points outside ANY bound appear as black "Discarded" markers and red dashed lines.
# Uncomment and fill in the bounds you want:
# OBJECTIVES_BOUNDS = {
#     "obj:max(μ(product_concentration))": {"min": 1000, "max": None},
#     "obj:max(R(product_concentration))": {"min": 1000,   "max": None},
#     "obj:max(P(purity))":                {"min": 0.6,  "max": None},
# }
OBJECTIVES_BOUNDS: dict[str, dict] = {}

PARETO_MODE = "Optimization"  # "Optimization" or "Manufacturing"

x_axis_names = ["μ(Titer) [mg/L]", "μ(Titer) [mg/L]", "μ(Titer) [mg/L]"]
y_axis_names = ["σ(Titer) [mg/L]", "Robustness(Titer) [mg/L]", "P(Purity) [-]"]
subplot_titles = ["Mean Prediction vs. Standard Deviation", "Mean Prediction vs. Robustness", "Mean Prediction vs. Constraint Probability"]


In [58]:
import pandas as pd

client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
runs = client.search_runs(experiment_ids=[experiment.experiment_id])
selected_run = next(r for r in runs if r.info.run_name == MLFLOW_RUN)
run_id = selected_run.info.run_id

temp_dir = os.path.join(LOCAL_TEMP_DIR, MLFLOW_RUN)
os.makedirs(temp_dir, exist_ok=True)

client.download_artifacts(run_id=run_id, path="general", dst_path=temp_dir)
client.download_artifacts(run_id=run_id, path=ITERATION, dst_path=temp_dir)

print(f"Artifacts downloaded to: {temp_dir}")

# Show objective column names — use these as keys in OBJECTIVES_BOUNDS
_pareto_csv = f"{temp_dir}/{ITERATION}/pareto_front_predictions.csv"
_obj_cols = [c for c in pd.read_csv(_pareto_csv, nrows=0).columns if c.startswith("obj:")]
print("\nObjective columns (use as OBJECTIVES_BOUNDS keys):")
for c in _obj_cols:
    print(f"  {c!r}")


2026-10-05 16:20:19,454 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:19,454 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.

Artifacts downloaded to: .temp/HIDL_8D

Objective columns (use as OBJECTIVES_BOUNDS keys):
  'obj:max(μ(product_concentration))'
  'obj:max(σ(product_concentration))'
  'obj:max(R(product_concentration))'
  'obj:max(P(purity))'


In [59]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_full = update_pareto_front_plots(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    clicked_points={},
    objectives_bounds=OBJECTIVES_BOUNDS,
    pinned_points={},
    objective_renames=OBJECTIVE_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    measurement_renames=MEASUREMENT_RENAMES,
    pareto_mode=PARETO_MODE,
)

# ── Row 1 only ────────────────────────────────────────────────────────────────
layout_dict = fig_full.to_plotly_json()["layout"]

# Row 1 yaxes: domain[1] ≈ 1.0 (topmost row always reaches the top)
row1_y_refs = {
    "y" + key[5:]
    for key, val in layout_dict.items()
    if re.match(r"^yaxis\d*$", key)
    and isinstance(val, dict)
    and abs((val.get("domain") or [0, 0])[1] - 1.0) < 0.01
}
row1_x_refs = {"x" + ref[1:] for ref in row1_y_refs}

# Map axis ref → subplot column (sorted: 'x'→1, 'x2'→2, 'x3'→3)
def _col_num(ref: str) -> int:
    n = ref[1:]  # strip leading 'x'
    return int(n) if n else 1

sorted_x_refs = sorted(row1_x_refs, key=_col_num)
x_ref_to_col = {ref: i + 1 for i, ref in enumerate(sorted_x_refs)}
n_cols = len(sorted_x_refs)

# Build proper 1-row subplot figure
fig = make_subplots(rows=1, cols=n_cols, horizontal_spacing=0.12, subplot_titles=subplot_titles[:n_cols])
fig.update_layout(template=fig_full.layout.template)

# Add row-1 traces with explicit col so xaxis/yaxis are always set correctly
for trace in fig_full.data:
    x_ref = trace.xaxis or "x"
    y_ref = trace.yaxis or "y"
    if y_ref in row1_y_refs:
        col = x_ref_to_col.get(x_ref, 1)
        fig.add_trace(trace, row=1, col=col)

# Copy axis titles/ticks from original (skip domain; make_subplots handles it)
for key, val in layout_dict.items():
    if not isinstance(val, dict):
        continue
    if re.match(r"^xaxis\d*$", key) and "x" + key[5:] in row1_x_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})
    elif re.match(r"^yaxis\d*$", key) and "y" + key[5:] in row1_y_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})

# Copy row-1 shapes (bounds vlines/hlines) using subplot-aware placement
for shape in fig_full.layout.shapes:
    s = shape.to_plotly_json()
    xref_base = (s.get("xref") or "").split(" ")[0]
    if xref_base not in row1_x_refs:
        continue
    col = x_ref_to_col.get(xref_base, 1)
    line = s.get("line", {})
    x0, x1 = s.get("x0"), s.get("x1")
    y0, y1 = s.get("y0"), s.get("y1")
    if x0 is not None and x0 == x1:          # vertical bound line
        fig.add_vline(x=float(x0), row=1, col=col, line=line)
    elif y0 is not None and y0 == y1:         # horizontal bound line
        fig.add_hline(y=float(y0), row=1, col=col, line=line)
    else:                                     # diagonal or other shape
        fig.add_shape(
            **{k: v for k, v in s.items() if k not in ("xref", "yref")},
            row=1, col=col,
        )

fig.update_layout(
    showlegend=layout_dict.get("showlegend", True),
    hovermode=layout_dict.get("hovermode"),
    legend=layout_dict.get("legend"),
    width=layout_dict.get("width"),
    height=500,
)

# ── Diagonal gray dashed line on second subplot (col 2) ───────────────────────
x2_vals, y2_vals = [], []
for trace in fig.data:
    if trace.xaxis == "x2" and trace.yaxis == "y2":
        if getattr(trace, "x", None) is not None:
            x2_vals.extend(v for v in trace.x if v is not None)
        if getattr(trace, "y", None) is not None:
            y2_vals.extend(v for v in trace.y if v is not None)

if x2_vals and y2_vals:
    diag_min = min(min(x2_vals), min(y2_vals))
    diag_max = max(max(x2_vals), max(y2_vals))
    fig.add_shape(
        type="line",
        x0=diag_min, y0=diag_min,
        x1=diag_max, y1=diag_max,
        row=1, col=2,
        line=dict(color="gray", width=2, dash="dash"),
    )

fig.update_layout(showlegend=False, margin=dict(l=240, r=20, t=110, b=120), width=2200, height=600,legend=dict(font=dict(size=30)))
for col_idx, (x_name, y_name) in enumerate(zip(x_axis_names, y_axis_names), start=1):
    fig.update_xaxes(title_text=x_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30),)
    fig.update_yaxes(title_text=y_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30))
for annotation in fig.layout.annotations:
    annotation.font.size = 25

# Iteration label on the left, spanning the whole row
fig.add_annotation(
    text=f"<b>Iteration {int(ITERATION.split('_')[-1])}</b>",
    x=-0.09, y=0.5,
    xref="paper", yref="paper",
    textangle=-90, xanchor="center", yanchor="middle",
    showarrow=False,
    font=dict(size=32),
)
fig.show()
fig.write_image(f"{PDF_FOLDER}/pareto_front_it2_all.pdf")


2026-10-05 16:20:19,624 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:19,625 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-10-05 16:20:19,696 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:19,697 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:19,697 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpesxuurrg.
2026-10-05 16:20:19,698 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:19,698 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpt53ik158.
2026-10-05 16:20:19,699 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpt53ik158
2026-10-05 16:20:20,130 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpesxuurrg/index.html
2026-10-05 16:20:20,708 - kaleido.kaleido - INFO - Getting tab from queue 

In [60]:
import pandas as pd

# ── Recipe selection (0-based row indices in the Pareto front CSV) ─────────────
EXPERIMENT_RECIPES = []   # shown as orange diamonds,  e.g. [0, 5, 12]
HIGHLIGHT_RECIPES  = [4,94, 126]   # shown as coloured X marks, e.g. [3, 7]

# ── Colour palette for highlight X marks (one colour per recipe) ──────────────
HIGHLIGHT_COLORS = [
    "#E72031", "#E11CF3", "#E9AC11", "#EC720E",
    "#05374B", "#7AF6FA", "#003A91", "#030912",
]

# ─────────────────────────────────────────────────────────────────────────────
pareto_df = pd.read_csv(
    f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/{ITERATION}/pareto_front_predictions.csv"
)
obj_cols = [c for c in pareto_df.columns if c.startswith("obj:")]

num_objectives = len(obj_cols)
all_pairs = [(i, j) for i in range(num_objectives) for j in range(i + 1, num_objectives)]
n_cols = min(3, len(all_pairs))
row1_pairs = all_pairs[:n_cols]
axis_refs = [("x", "y"), ("x2", "y2"), ("x3", "y3")][:n_cols]

for subplot_idx, ((i, j), (xref, yref)) in enumerate(zip(row1_pairs, axis_refs)):
    x_col, y_col = obj_cols[i], obj_cols[j]

    # Orange diamonds
    if EXPERIMENT_RECIPES:
        sel = pareto_df.iloc[EXPERIMENT_RECIPES]
        fig.add_trace(go.Scatter(
            x=sel[x_col], y=sel[y_col],
            mode="markers",
            marker=dict(
                size=13,
                symbol="diamond",
                color="#FFA500",
                line=dict(width=2, color="black"),
            ),
            name="Experiments",
            legendgroup="experiments",
            showlegend=(subplot_idx == 0),
            xaxis=xref, yaxis=yref,
        ))

    # Coloured X marks — one trace per recipe so each gets its own legend entry
    for color_idx, rec_idx in enumerate(HIGHLIGHT_RECIPES):
        row = pareto_df.iloc[rec_idx]
        fig.add_trace(go.Scatter(
            x=[row[x_col]], y=[row[y_col]],
            mode="markers",
            marker=dict(
                size=14,
                symbol="x",
                color=HIGHLIGHT_COLORS[color_idx % len(HIGHLIGHT_COLORS)],
                line=dict(width=3),
            ),
            name=f"Recipe #{rec_idx}",
            legendgroup=f"highlight_{rec_idx}",
            showlegend=(subplot_idx == 0),
            xaxis=xref, yaxis=yref,
        ))
fig.update_layout(showlegend=True, legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.375), margin=dict(l=200, r=20, t=50, b=120), width=2320, height=600)
fig.show()
fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.write_image(f"{PDF_FOLDER}/results_{ITERATION}.pdf")


2026-10-05 16:20:20,982 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:20,982 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-10-05 16:20:20,991 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:20,992 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:20,993 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmptpj7z5ny.
2026-10-05 16:20:20,993 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:20,994 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpw6jrams4.
2026-10-05 16:20:20,994 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpw6jrams4
2026-10-05 16:20:21,414 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmptpj7z5ny/index.html
2026-10-05 16:20:22,020 - kaleido.kaleido - INFO - Getting tab from queue 

# Iteration 7

In [61]:
# MLflow tracking URI
# Local SQLite:  "sqlite:////absolute/path/to/mlflow.db"
# Running server: "http://127.0.0.1:5000"
MLFLOW_TRACKING_URI = "sqlite:////Users/ss15425/Documents/00_Git_Repositories/pfgs-research/mlflow.db"

MLFLOW_EXPERIMENT = "Journal-02-06-2026"
MLFLOW_RUN = "HIDL_8D"
ITERATION = "Iteration_007"

# Renames: map internal column names to display labels (leave empty for defaults)
OBJECTIVE_RENAMES: dict[str, str] = {}
INVESTIGATION_SPACE_RENAMES: dict[str, str] = {}
MEASUREMENT_RENAMES: dict[str, str] = {}

# Bounds — keys must match the exact objective column names printed by cell 3.
# Points outside ANY bound appear as black "Discarded" markers and red dashed lines.
# Uncomment and fill in the bounds you want:
OBJECTIVES_BOUNDS = {
    "obj:max(σ(product_concentration))": {"min": 100, "max": None},
    "obj:max(R(product_concentration))": {"min": 1000,   "max": None},
    "obj:max(P(purity))":                {"min": 0.9,  "max": None},
}
# OBJECTIVES_BOUNDS: dict[str, dict] = {}

PARETO_MODE = "Optimization"  # "Optimization" or "Manufacturing"

x_axis_names = ["μ(Titer) [mg/L]", "μ(Titer) [mg/L]", "μ(Titer) [mg/L]"]
y_axis_names = ["σ(Titer) [mg/L]", "Robustness(Titer) [mg/L]", "P(Purity) [-]"]
subplot_titles = ["Mean Prediction vs. Standard Deviation", "Mean Prediction vs. Robustness", "Mean Prediction vs. Constraint Probability"]


In [62]:
import pandas as pd

client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
runs = client.search_runs(experiment_ids=[experiment.experiment_id])
selected_run = next(r for r in runs if r.info.run_name == MLFLOW_RUN)
run_id = selected_run.info.run_id

temp_dir = os.path.join(LOCAL_TEMP_DIR, MLFLOW_RUN)
os.makedirs(temp_dir, exist_ok=True)

client.download_artifacts(run_id=run_id, path="general", dst_path=temp_dir)
client.download_artifacts(run_id=run_id, path=ITERATION, dst_path=temp_dir)

print(f"Artifacts downloaded to: {temp_dir}")

# Show objective column names — use these as keys in OBJECTIVES_BOUNDS
_pareto_csv = f"{temp_dir}/{ITERATION}/pareto_front_predictions.csv"
_obj_cols = [c for c in pd.read_csv(_pareto_csv, nrows=0).columns if c.startswith("obj:")]
print("\nObjective columns (use as OBJECTIVES_BOUNDS keys):")
for c in _obj_cols:
    print(f"  {c!r}")


2026-10-05 16:20:22,397 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:22,398 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.

Artifacts downloaded to: .temp/HIDL_8D

Objective columns (use as OBJECTIVES_BOUNDS keys):
  'obj:max(μ(product_concentration))'
  'obj:max(σ(product_concentration))'
  'obj:max(R(product_concentration))'
  'obj:max(P(purity))'


In [63]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_full = update_pareto_front_plots(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    clicked_points={},
    objectives_bounds=OBJECTIVES_BOUNDS,
    pinned_points={},
    objective_renames=OBJECTIVE_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    measurement_renames=MEASUREMENT_RENAMES,
    pareto_mode=PARETO_MODE,
)

# ── Row 1 only ────────────────────────────────────────────────────────────────
layout_dict = fig_full.to_plotly_json()["layout"]

# Row 1 yaxes: domain[1] ≈ 1.0 (topmost row always reaches the top)
row1_y_refs = {
    "y" + key[5:]
    for key, val in layout_dict.items()
    if re.match(r"^yaxis\d*$", key)
    and isinstance(val, dict)
    and abs((val.get("domain") or [0, 0])[1] - 1.0) < 0.01
}
row1_x_refs = {"x" + ref[1:] for ref in row1_y_refs}

# Map axis ref → subplot column (sorted: 'x'→1, 'x2'→2, 'x3'→3)
def _col_num(ref: str) -> int:
    n = ref[1:]  # strip leading 'x'
    return int(n) if n else 1

sorted_x_refs = sorted(row1_x_refs, key=_col_num)
x_ref_to_col = {ref: i + 1 for i, ref in enumerate(sorted_x_refs)}
n_cols = len(sorted_x_refs)

# Build proper 1-row subplot figure
fig = make_subplots(rows=1, cols=n_cols, horizontal_spacing=0.12, subplot_titles=subplot_titles[:n_cols])
fig.update_layout(template=fig_full.layout.template)

# Add row-1 traces with explicit col so xaxis/yaxis are always set correctly
for trace in fig_full.data:
    x_ref = trace.xaxis or "x"
    y_ref = trace.yaxis or "y"
    if y_ref in row1_y_refs:
        col = x_ref_to_col.get(x_ref, 1)
        fig.add_trace(trace, row=1, col=col)

# Copy axis titles/ticks from original (skip domain; make_subplots handles it)
for key, val in layout_dict.items():
    if not isinstance(val, dict):
        continue
    if re.match(r"^xaxis\d*$", key) and "x" + key[5:] in row1_x_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})
    elif re.match(r"^yaxis\d*$", key) and "y" + key[5:] in row1_y_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})

# Copy row-1 shapes (bounds vlines/hlines) using subplot-aware placement
for shape in fig_full.layout.shapes:
    s = shape.to_plotly_json()
    xref_base = (s.get("xref") or "").split(" ")[0]
    if xref_base not in row1_x_refs:
        continue
    col = x_ref_to_col.get(xref_base, 1)
    line = s.get("line", {})
    x0, x1 = s.get("x0"), s.get("x1")
    y0, y1 = s.get("y0"), s.get("y1")
    if x0 is not None and x0 == x1:          # vertical bound line
        fig.add_vline(x=float(x0), row=1, col=col, line=line)
    elif y0 is not None and y0 == y1:         # horizontal bound line
        fig.add_hline(y=float(y0), row=1, col=col, line=line)
    else:                                     # diagonal or other shape
        fig.add_shape(
            **{k: v for k, v in s.items() if k not in ("xref", "yref")},
            row=1, col=col,
        )

fig.update_layout(
    showlegend=layout_dict.get("showlegend", True),
    hovermode=layout_dict.get("hovermode"),
    legend=layout_dict.get("legend"),
    width=layout_dict.get("width"),
    height=500,
)

# ── Diagonal gray dashed line on second subplot (col 2) ───────────────────────
x2_vals, y2_vals = [], []
for trace in fig.data:
    if trace.xaxis == "x2" and trace.yaxis == "y2":
        if getattr(trace, "x", None) is not None:
            x2_vals.extend(v for v in trace.x if v is not None)
        if getattr(trace, "y", None) is not None:
            y2_vals.extend(v for v in trace.y if v is not None)

if x2_vals and y2_vals:
    diag_min = min(min(x2_vals), min(y2_vals))
    diag_max = max(max(x2_vals), max(y2_vals))
    fig.add_shape(
        type="line",
        x0=diag_min, y0=diag_min,
        x1=diag_max, y1=diag_max,
        row=1, col=2,
        line=dict(color="gray", width=2, dash="dash"),
    )

fig.update_layout(showlegend=False, margin=dict(l=240, r=20, t=110, b=120), width=2200, height=600,legend=dict(font=dict(size=30)))
for col_idx, (x_name, y_name) in enumerate(zip(x_axis_names, y_axis_names), start=1):
    fig.update_xaxes(title_text=x_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30),)
    fig.update_yaxes(title_text=y_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30))
for annotation in fig.layout.annotations:
    annotation.font.size = 25

# Iteration label on the left, spanning the whole row
fig.add_annotation(
    text=f"<b>Iteration {int(ITERATION.split('_')[-1])}</b>",
    x=-0.09, y=0.5,
    xref="paper", yref="paper",
    textangle=-90, xanchor="center", yanchor="middle",
    showarrow=False,
    font=dict(size=32),
)
fig.show()


In [64]:
import pandas as pd

# ── Recipe selection (0-based row indices in the Pareto front CSV) ─────────────
EXPERIMENT_RECIPES = []   # shown as orange diamonds,  e.g. [0, 5, 12]
HIGHLIGHT_RECIPES  = []   # shown as coloured X marks, e.g. [3, 7]

# ── Colour palette for highlight X marks (one colour per recipe) ──────────────
HIGHLIGHT_COLORS = [
    "#E72031", "#108173", "#E9AC11", "#EC720E",
    "#05374B", "#7AF6FA", "#003A91", "#030912",
]

# ─────────────────────────────────────────────────────────────────────────────
pareto_df = pd.read_csv(
    f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/{ITERATION}/pareto_front_predictions.csv"
)
obj_cols = [c for c in pareto_df.columns if c.startswith("obj:")]

num_objectives = len(obj_cols)
all_pairs = [(i, j) for i in range(num_objectives) for j in range(i + 1, num_objectives)]
n_cols = min(3, len(all_pairs))
row1_pairs = all_pairs[:n_cols]
axis_refs = [("x", "y"), ("x2", "y2"), ("x3", "y3")][:n_cols]

for subplot_idx, ((i, j), (xref, yref)) in enumerate(zip(row1_pairs, axis_refs)):
    x_col, y_col = obj_cols[i], obj_cols[j]

    # Orange diamonds
    if EXPERIMENT_RECIPES:
        sel = pareto_df.iloc[EXPERIMENT_RECIPES]
        fig.add_trace(go.Scatter(
            x=sel[x_col], y=sel[y_col],
            mode="markers",
            marker=dict(
                size=13,
                symbol="diamond",
                color="#FFA500",
                line=dict(width=2, color="black"),
            ),
            name="Experiments",
            legendgroup="experiments",
            showlegend=(subplot_idx == 0),
            xaxis=xref, yaxis=yref,
        ))

    # Coloured X marks — one trace per recipe so each gets its own legend entry
    for color_idx, rec_idx in enumerate(HIGHLIGHT_RECIPES):
        row = pareto_df.iloc[rec_idx]
        fig.add_trace(go.Scatter(
            x=[row[x_col]], y=[row[y_col]],
            mode="markers",
            marker=dict(
                size=14,
                symbol="x",
                color=HIGHLIGHT_COLORS[color_idx % len(HIGHLIGHT_COLORS)],
                line=dict(width=3),
            ),
            name=f"Recipe #{rec_idx}",
            legendgroup=f"highlight_{rec_idx}",
            showlegend=(subplot_idx == 0),
            xaxis=xref, yaxis=yref,
        ))
fig.update_layout(showlegend=True, legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.375), margin=dict(l=200, r=20, t=50, b=120), width=2320, height=600)
fig.show()
fig.update_layout(
    paper_bgcolor='rgba(0,0,0,0)'
)
fig.write_image(f"{PDF_FOLDER}/results_{ITERATION}.pdf")


2026-10-05 16:20:22,581 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:22,583 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:22,584 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpk08hyu2_.
2026-10-05 16:20:22,585 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:22,586 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp699al3uu.
2026-10-05 16:20:22,587 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp699al3uu
2026-10-05 16:20:23,100 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpk08hyu2_/index.html
2026-10-05 16:20:23,750 - kaleido.kaleido - INFO - Getting tab from queue 

# 2D for Methodology Section

In [65]:
# MLflow tracking URI
# Local SQLite:  "sqlite:////absolute/path/to/mlflow.db"
# Running server: "http://127.0.0.1:5000"
MLFLOW_TRACKING_URI = "sqlite:////Users/ss15425/Documents/00_Git_Repositories/pfgs-research/mlflow.db"

MLFLOW_EXPERIMENT = "Journal-02-06-2026"
MLFLOW_RUN = "Obj"
ITERATION = "Iteration_001"

# Renames: map internal column names to display labels (leave empty for defaults)
OBJECTIVE_RENAMES: dict[str, str] = {}
INVESTIGATION_SPACE_RENAMES: dict[str, str] = {}
MEASUREMENT_RENAMES: dict[str, str] = {}

# Bounds — keys must match the exact objective column names printed by cell 3.
# Points outside ANY bound appear as black "Discarded" markers and red dashed lines.
# Uncomment and fill in the bounds you want:
OBJECTIVES_BOUNDS = {
    # "obj:max(σ(product_concentration))": {"min": 250, "max": None},
    # "obj:max(μ(product_concentration))": {"min": 1600,   "max": None},
}
# OBJECTIVES_BOUNDS: dict[str, dict] = {}

PARETO_MODE = "Optimization"  # "Optimization" or "Manufacturing"

x_axis_names = ["μ(Titer) [mg/L]", "μ(Titer) [mg/L]", "μ(Titer) [mg/L]"]
y_axis_names = ["σ(Titer) [mg/L]", "Robustness(Titer) [mg/L]", "P(Purity) [-]"]


In [66]:
import pandas as pd

client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
runs = client.search_runs(experiment_ids=[experiment.experiment_id])
selected_run = next(r for r in runs if r.info.run_name == MLFLOW_RUN)
run_id = selected_run.info.run_id

temp_dir = os.path.join(LOCAL_TEMP_DIR, MLFLOW_RUN)
os.makedirs(temp_dir, exist_ok=True)

client.download_artifacts(run_id=run_id, path="general", dst_path=temp_dir)
client.download_artifacts(run_id=run_id, path=ITERATION, dst_path=temp_dir)

print(f"Artifacts downloaded to: {temp_dir}")

# Show objective column names — use these as keys in OBJECTIVES_BOUNDS
_pareto_csv = f"{temp_dir}/{ITERATION}/pareto_front_predictions.csv"
_obj_cols = [c for c in pd.read_csv(_pareto_csv, nrows=0).columns if c.startswith("obj:")]
print("\nObjective columns (use as OBJECTIVES_BOUNDS keys):")
for c in _obj_cols:
    print(f"  {c!r}")


2026-10-05 16:20:24,040 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:24,040 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


Artifacts downloaded to: .temp/Obj

Objective columns (use as OBJECTIVES_BOUNDS keys):
  'obj:max(μ(product_concentration))'
  'obj:max(σ(product_concentration))'


In [67]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_full = update_pareto_front_plots(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    clicked_points={},
    objectives_bounds=OBJECTIVES_BOUNDS,
    pinned_points={},
    objective_renames=OBJECTIVE_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    measurement_renames=MEASUREMENT_RENAMES,
    pareto_mode=PARETO_MODE,
)

# ── Row 1 only ────────────────────────────────────────────────────────────────
layout_dict = fig_full.to_plotly_json()["layout"]

# Row 1 yaxes: domain[1] ≈ 1.0 (topmost row always reaches the top)
row1_y_refs = {
    "y" + key[5:]
    for key, val in layout_dict.items()
    if re.match(r"^yaxis\d*$", key)
    and isinstance(val, dict)
    and abs((val.get("domain") or [0, 0])[1] - 1.0) < 0.01
}
row1_x_refs = {"x" + ref[1:] for ref in row1_y_refs}

# Map axis ref → subplot column (sorted: 'x'→1, 'x2'→2, 'x3'→3)
def _col_num(ref: str) -> int:
    n = ref[1:]  # strip leading 'x'
    return int(n) if n else 1

sorted_x_refs = sorted(row1_x_refs, key=_col_num)
x_ref_to_col = {ref: i + 1 for i, ref in enumerate(sorted_x_refs)}
n_cols = len(sorted_x_refs)

# Build proper 1-row subplot figure
fig = make_subplots(rows=1, cols=n_cols, horizontal_spacing=0.12)
fig.update_layout(template=fig_full.layout.template)

# Add row-1 traces with explicit col so xaxis/yaxis are always set correctly
for trace in fig_full.data:
    x_ref = trace.xaxis or "x"
    y_ref = trace.yaxis or "y"
    if y_ref in row1_y_refs:
        col = x_ref_to_col.get(x_ref, 1)
        fig.add_trace(trace, row=1, col=col)

# Copy axis titles/ticks from original (skip domain; make_subplots handles it)
for key, val in layout_dict.items():
    if not isinstance(val, dict):
        continue
    if re.match(r"^xaxis\d*$", key) and "x" + key[5:] in row1_x_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})
    elif re.match(r"^yaxis\d*$", key) and "y" + key[5:] in row1_y_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})

# Copy row-1 shapes (bounds vlines/hlines) using subplot-aware placement
for shape in fig_full.layout.shapes:
    s = shape.to_plotly_json()
    xref_base = (s.get("xref") or "").split(" ")[0]
    if xref_base not in row1_x_refs:
        continue
    col = x_ref_to_col.get(xref_base, 1)
    line = s.get("line", {})
    x0, x1 = s.get("x0"), s.get("x1")
    y0, y1 = s.get("y0"), s.get("y1")
    if x0 is not None and x0 == x1:          # vertical bound line
        fig.add_vline(x=float(x0), row=1, col=col, line=line)
    elif y0 is not None and y0 == y1:         # horizontal bound line
        fig.add_hline(y=float(y0), row=1, col=col, line=line)
    else:                                     # diagonal or other shape
        fig.add_shape(
            **{k: v for k, v in s.items() if k not in ("xref", "yref")},
            row=1, col=col,
        )

fig.update_layout(
    showlegend=layout_dict.get("showlegend", True),
    hovermode=layout_dict.get("hovermode"),
    legend=layout_dict.get("legend"),
    width=layout_dict.get("width"),
    height=500,
)

# ── Diagonal gray dashed line on second subplot (col 2) ───────────────────────
x2_vals, y2_vals = [], []
for trace in fig.data:
    if trace.xaxis == "x2" and trace.yaxis == "y2":
        if getattr(trace, "x", None) is not None:
            x2_vals.extend(v for v in trace.x if v is not None)
        if getattr(trace, "y", None) is not None:
            y2_vals.extend(v for v in trace.y if v is not None)

if x2_vals and y2_vals:
    diag_min = min(min(x2_vals), min(y2_vals))
    diag_max = max(max(x2_vals), max(y2_vals))
    fig.add_shape(
        type="line",
        x0=diag_min, y0=diag_min,
        x1=diag_max, y1=diag_max,
        row=1, col=2,
        line=dict(color="gray", width=2, dash="dash"),
    )

fig.update_layout(showlegend=True, margin=dict(l=100, r=20, t=130, b=100), width=600, height=600,legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99, font=dict(size=30)))
for col_idx, (x_name, y_name) in enumerate(zip(x_axis_names, y_axis_names), start=1):
    fig.update_xaxes(title_text=x_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30),)
    fig.update_yaxes(title_text=y_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30))
fig.update_layout(title=dict(text="Pareto Front", font=dict(size=22), x=0.5, xanchor="center"))
fig.show()
fig_b=fig
fig.write_image(f"{PDF_FOLDER}/init_ms_pareto-front.pdf")

2026-10-05 16:20:24,096 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:24,097 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:24,098 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpqss47vpw.
2026-10-05 16:20:24,098 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:24,099 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpm0a8osyi.
2026-10-05 16:20:24,099 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpm0a8osyi
2026-10-05 16:20:24,497 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpqss47vpw/index.html
2026-10-05 16:20:25,122 - kaleido.kaleido - INFO - Getting tab from queue 

In [68]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots
OBJECTIVES_BOUNDS = {
    # "obj:max(σ(product_concentration))": {"min": 250, "max": None},
    "obj:max(μ(product_concentration))": {"min": 1600,   "max": None},
}
fig_full = update_pareto_front_plots(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    clicked_points={},
    objectives_bounds=OBJECTIVES_BOUNDS,
    pinned_points={},
    objective_renames=OBJECTIVE_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    measurement_renames=MEASUREMENT_RENAMES,
    pareto_mode=PARETO_MODE,
)

# ── Row 1 only ────────────────────────────────────────────────────────────────
layout_dict = fig_full.to_plotly_json()["layout"]

# Row 1 yaxes: domain[1] ≈ 1.0 (topmost row always reaches the top)
row1_y_refs = {
    "y" + key[5:]
    for key, val in layout_dict.items()
    if re.match(r"^yaxis\d*$", key)
    and isinstance(val, dict)
    and abs((val.get("domain") or [0, 0])[1] - 1.0) < 0.01
}
row1_x_refs = {"x" + ref[1:] for ref in row1_y_refs}

# Map axis ref → subplot column (sorted: 'x'→1, 'x2'→2, 'x3'→3)
def _col_num(ref: str) -> int:
    n = ref[1:]  # strip leading 'x'
    return int(n) if n else 1

sorted_x_refs = sorted(row1_x_refs, key=_col_num)
x_ref_to_col = {ref: i + 1 for i, ref in enumerate(sorted_x_refs)}
n_cols = len(sorted_x_refs)

# Build proper 1-row subplot figure
fig = make_subplots(rows=1, cols=n_cols, horizontal_spacing=0.12)
fig.update_layout(template=fig_full.layout.template)

# Add row-1 traces with explicit col so xaxis/yaxis are always set correctly
for trace in fig_full.data:
    x_ref = trace.xaxis or "x"
    y_ref = trace.yaxis or "y"
    if y_ref in row1_y_refs:
        col = x_ref_to_col.get(x_ref, 1)
        fig.add_trace(trace, row=1, col=col)

# Copy axis titles/ticks from original (skip domain; make_subplots handles it)
for key, val in layout_dict.items():
    if not isinstance(val, dict):
        continue
    if re.match(r"^xaxis\d*$", key) and "x" + key[5:] in row1_x_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})
    elif re.match(r"^yaxis\d*$", key) and "y" + key[5:] in row1_y_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})

# Copy row-1 shapes (bounds vlines/hlines) using subplot-aware placement
for shape in fig_full.layout.shapes:
    s = shape.to_plotly_json()
    xref_base = (s.get("xref") or "").split(" ")[0]
    if xref_base not in row1_x_refs:
        continue
    col = x_ref_to_col.get(xref_base, 1)
    line = s.get("line", {})
    x0, x1 = s.get("x0"), s.get("x1")
    y0, y1 = s.get("y0"), s.get("y1")
    if x0 is not None and x0 == x1:          # vertical bound line
        fig.add_vline(x=float(x0), row=1, col=col, line=line)
    elif y0 is not None and y0 == y1:         # horizontal bound line
        fig.add_hline(y=float(y0), row=1, col=col, line=line)
    else:                                     # diagonal or other shape
        fig.add_shape(
            **{k: v for k, v in s.items() if k not in ("xref", "yref")},
            row=1, col=col,
        )

fig.update_layout(
    showlegend=layout_dict.get("showlegend", True),
    hovermode=layout_dict.get("hovermode"),
    legend=layout_dict.get("legend"),
    width=layout_dict.get("width"),
    height=500,
)

# ── Diagonal gray dashed line on second subplot (col 2) ───────────────────────
x2_vals, y2_vals = [], []
for trace in fig.data:
    if trace.xaxis == "x2" and trace.yaxis == "y2":
        if getattr(trace, "x", None) is not None:
            x2_vals.extend(v for v in trace.x if v is not None)
        if getattr(trace, "y", None) is not None:
            y2_vals.extend(v for v in trace.y if v is not None)

if x2_vals and y2_vals:
    diag_min = min(min(x2_vals), min(y2_vals))
    diag_max = max(max(x2_vals), max(y2_vals))
    fig.add_shape(
        type="line",
        x0=diag_min, y0=diag_min,
        x1=diag_max, y1=diag_max,
        row=1, col=2,
        line=dict(color="gray", width=2, dash="dash"),
    )

fig.update_layout(showlegend=True, margin=dict(l=100, r=20, t=130, b=110), width=600, height=600,legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99, font=dict(size=30)))
for col_idx, (x_name, y_name) in enumerate(zip(x_axis_names, y_axis_names), start=1):
    fig.update_xaxes(title_text=x_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30),)
    fig.update_yaxes(title_text=y_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30))
fig.update_layout(title=dict(text="Expert Limit on μ(Titer)", font=dict(size=22), x=0.5, xanchor="center"))
fig.show()
fig.write_image(f"{PDF_FOLDER}/pareto-front-mu-bounds.pdf")

2026-10-05 16:20:25,368 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:25,368 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-10-05 16:20:25,390 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:25,391 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:25,392 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmploadflaw.
2026-10-05 16:20:25,392 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:25,393 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmph5mv1t6c.
2026-10-05 16:20:25,394 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmph5mv1t6c
2026-10-05 16:20:25,831 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmploadflaw/index.html
2026-10-05 16:20:26,413 - kaleido.kaleido - INFO - Getting tab from queue 

In [69]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots
OBJECTIVES_BOUNDS = {
    "obj:max(σ(product_concentration))": {"min": 250, "max": None},
    "obj:max(μ(product_concentration))": {"min": 1600,   "max": None},
}
fig_full = update_pareto_front_plots(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    clicked_points={},
    objectives_bounds=OBJECTIVES_BOUNDS,
    pinned_points={},
    objective_renames=OBJECTIVE_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    measurement_renames=MEASUREMENT_RENAMES,
    pareto_mode=PARETO_MODE,
)

# ── Row 1 only ────────────────────────────────────────────────────────────────
layout_dict = fig_full.to_plotly_json()["layout"]

# Row 1 yaxes: domain[1] ≈ 1.0 (topmost row always reaches the top)
row1_y_refs = {
    "y" + key[5:]
    for key, val in layout_dict.items()
    if re.match(r"^yaxis\d*$", key)
    and isinstance(val, dict)
    and abs((val.get("domain") or [0, 0])[1] - 1.0) < 0.01
}
row1_x_refs = {"x" + ref[1:] for ref in row1_y_refs}

# Map axis ref → subplot column (sorted: 'x'→1, 'x2'→2, 'x3'→3)
def _col_num(ref: str) -> int:
    n = ref[1:]  # strip leading 'x'
    return int(n) if n else 1

sorted_x_refs = sorted(row1_x_refs, key=_col_num)
x_ref_to_col = {ref: i + 1 for i, ref in enumerate(sorted_x_refs)}
n_cols = len(sorted_x_refs)

# Build proper 1-row subplot figure
fig = make_subplots(rows=1, cols=n_cols, horizontal_spacing=0.12)
fig.update_layout(template=fig_full.layout.template)

# Add row-1 traces with explicit col so xaxis/yaxis are always set correctly
for trace in fig_full.data:
    x_ref = trace.xaxis or "x"
    y_ref = trace.yaxis or "y"
    if y_ref in row1_y_refs:
        col = x_ref_to_col.get(x_ref, 1)
        fig.add_trace(trace, row=1, col=col)

# Copy axis titles/ticks from original (skip domain; make_subplots handles it)
for key, val in layout_dict.items():
    if not isinstance(val, dict):
        continue
    if re.match(r"^xaxis\d*$", key) and "x" + key[5:] in row1_x_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})
    elif re.match(r"^yaxis\d*$", key) and "y" + key[5:] in row1_y_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})

# Copy row-1 shapes (bounds vlines/hlines) using subplot-aware placement
for shape in fig_full.layout.shapes:
    s = shape.to_plotly_json()
    xref_base = (s.get("xref") or "").split(" ")[0]
    if xref_base not in row1_x_refs:
        continue
    col = x_ref_to_col.get(xref_base, 1)
    line = s.get("line", {})
    x0, x1 = s.get("x0"), s.get("x1")
    y0, y1 = s.get("y0"), s.get("y1")
    if x0 is not None and x0 == x1:          # vertical bound line
        fig.add_vline(x=float(x0), row=1, col=col, line=line)
    elif y0 is not None and y0 == y1:         # horizontal bound line
        fig.add_hline(y=float(y0), row=1, col=col, line=line)
    else:                                     # diagonal or other shape
        fig.add_shape(
            **{k: v for k, v in s.items() if k not in ("xref", "yref")},
            row=1, col=col,
        )

fig.update_layout(
    showlegend=layout_dict.get("showlegend", True),
    hovermode=layout_dict.get("hovermode"),
    legend=layout_dict.get("legend"),
    width=layout_dict.get("width"),
    height=500,
)

# ── Diagonal gray dashed line on second subplot (col 2) ───────────────────────
x2_vals, y2_vals = [], []
for trace in fig.data:
    if trace.xaxis == "x2" and trace.yaxis == "y2":
        if getattr(trace, "x", None) is not None:
            x2_vals.extend(v for v in trace.x if v is not None)
        if getattr(trace, "y", None) is not None:
            y2_vals.extend(v for v in trace.y if v is not None)

if x2_vals and y2_vals:
    diag_min = min(min(x2_vals), min(y2_vals))
    diag_max = max(max(x2_vals), max(y2_vals))
    fig.add_shape(
        type="line",
        x0=diag_min, y0=diag_min,
        x1=diag_max, y1=diag_max,
        row=1, col=2,
        line=dict(color="gray", width=2, dash="dash"),
    )

fig.update_layout(showlegend=True, margin=dict(l=100, r=20, t=130, b=110), width=600, height=600,legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99, font=dict(size=30)))
for col_idx, (x_name, y_name) in enumerate(zip(x_axis_names, y_axis_names), start=1):
    fig.update_xaxes(title_text=x_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30),)
    fig.update_yaxes(title_text=y_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30))
fig.update_layout(title=dict(text="Expert Limits on μ(Titer) and σ(Titer)", font=dict(size=22), x=0.5, xanchor="center"))
fig.show()
fig.write_image(f"{PDF_FOLDER}/pareto-front-sigma-mu-bounds.pdf")

2026-10-05 16:20:26,724 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:26,724 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-10-05 16:20:26,744 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:26,745 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:26,746 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpu_8iack2.
2026-10-05 16:20:26,747 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:26,747 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp9m1doout.
2026-10-05 16:20:26,748 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp9m1doout
2026-10-05 16:20:27,175 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpu_8iack2/index.html
2026-10-05 16:20:27,802 - kaleido.kaleido - INFO - Getting tab from queue 

In [70]:
import pandas as pd

# ── Recipe selection (0-based row indices in the Pareto front CSV) ─────────────
EXPERIMENT_RECIPES = [27,26,25,3,4]   # shown as orange diamonds,  e.g. [0, 5, 12]
HIGHLIGHT_RECIPES  = []   # shown as coloured X marks, e.g. [3, 7]

# ── Colour palette for highlight X marks (one colour per recipe) ──────────────
HIGHLIGHT_COLORS = [
    "#E72031", "#108173", "#E9AC11", "#EC720E",
    "#05374B", "#7AF6FA", "#003A91", "#030912",
]

# ─────────────────────────────────────────────────────────────────────────────
pareto_df = pd.read_csv(
    f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/{ITERATION}/pareto_front_predictions.csv"
)
obj_cols = [c for c in pareto_df.columns if c.startswith("obj:")]

num_objectives = len(obj_cols)
all_pairs = [(i, j) for i in range(num_objectives) for j in range(i + 1, num_objectives)]
n_cols = min(3, len(all_pairs))
row1_pairs = all_pairs[:n_cols]
axis_refs = [("x", "y"), ("x2", "y2"), ("x3", "y3")][:n_cols]

for subplot_idx, ((i, j), (xref, yref)) in enumerate(zip(row1_pairs, axis_refs)):
    x_col, y_col = obj_cols[i], obj_cols[j]

    # Orange diamonds
    if EXPERIMENT_RECIPES:
        sel = pareto_df.iloc[EXPERIMENT_RECIPES]
        fig.add_trace(go.Scatter(
            x=sel[x_col], y=sel[y_col],
            mode="markers",
            marker=dict(
                size=18,
                symbol="diamond",
                color="#FFA500",
                line=dict(width=2, color="black"),
            ),
            name="Experiments",
            legendgroup="experiments",
            showlegend=(subplot_idx == 0),
            xaxis=xref, yaxis=yref,
        ))

    # Coloured X marks — one trace per recipe so each gets its own legend entry
    for color_idx, rec_idx in enumerate(HIGHLIGHT_RECIPES):
        row = pareto_df.iloc[rec_idx]
        fig.add_trace(go.Scatter(
            x=[row[x_col]], y=[row[y_col]],
            mode="markers",
            marker=dict(
                size=14,
                symbol="x",
                color=HIGHLIGHT_COLORS[color_idx % len(HIGHLIGHT_COLORS)],
                line=dict(width=3),
            ),
            name=f"Recipe #{rec_idx}",
            legendgroup=f"highlight_{rec_idx}",
            showlegend=(subplot_idx == 0),
            xaxis=xref, yaxis=yref,
        ))
fig.update_layout(showlegend=True, legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99), margin=dict(l=100, r=20, t=50, b=110), width=600, height=600)
fig.show()
fig_d = fig
fig.write_image(f"{PDF_FOLDER}/init_ms_res_pareto-front.pdf")

2026-10-05 16:20:28,079 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:28,081 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:28,082 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp1hlld32e.
2026-10-05 16:20:28,083 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:28,084 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpi2rxodjq.
2026-10-05 16:20:28,084 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpi2rxodjq
2026-10-05 16:20:28,523 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp1hlld32e/index.html
2026-10-05 16:20:29,108 - kaleido.kaleido - INFO - Getting tab from queue 

## Iteration 2

In [71]:
# MLflow tracking URI
# Local SQLite:  "sqlite:////absolute/path/to/mlflow.db"
# Running server: "http://127.0.0.1:5000"
MLFLOW_TRACKING_URI = "sqlite:////Users/ss15425/Documents/00_Git_Repositories/pfgs-research/mlflow.db"

MLFLOW_EXPERIMENT = "Journal-02-06-2026"
MLFLOW_RUN = "Obj"
ITERATION = "Iteration_002"

# Renames: map internal column names to display labels (leave empty for defaults)
OBJECTIVE_RENAMES: dict[str, str] = {}
INVESTIGATION_SPACE_RENAMES: dict[str, str] = {}
MEASUREMENT_RENAMES: dict[str, str] = {}

# Bounds — keys must match the exact objective column names printed by cell 3.
# Points outside ANY bound appear as black "Discarded" markers and red dashed lines.
# Uncomment and fill in the bounds you want:
OBJECTIVES_BOUNDS = {
    # "obj:max(σ(product_concentration))": {"min": 250, "max": None},
    # "obj:max(μ(product_concentration))": {"min": 1600,   "max": None},
}
# OBJECTIVES_BOUNDS: dict[str, dict] = {}

PARETO_MODE = "Optimization"  # "Optimization" or "Manufacturing"

x_axis_names = ["μ(Titer) [mg/L]", "μ(Titer) [mg/L]", "μ(Titer) [mg/L]"]
y_axis_names = ["σ(Titer) [mg/L]", "Robustness(Titer) [mg/L]", "P(Purity) [-]"]


In [72]:
import pandas as pd

client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
runs = client.search_runs(experiment_ids=[experiment.experiment_id])
selected_run = next(r for r in runs if r.info.run_name == MLFLOW_RUN)
run_id = selected_run.info.run_id

temp_dir = os.path.join(LOCAL_TEMP_DIR, MLFLOW_RUN)
os.makedirs(temp_dir, exist_ok=True)

client.download_artifacts(run_id=run_id, path="general", dst_path=temp_dir)
client.download_artifacts(run_id=run_id, path=ITERATION, dst_path=temp_dir)

print(f"Artifacts downloaded to: {temp_dir}")

# Show objective column names — use these as keys in OBJECTIVES_BOUNDS
_pareto_csv = f"{temp_dir}/{ITERATION}/pareto_front_predictions.csv"
_obj_cols = [c for c in pd.read_csv(_pareto_csv, nrows=0).columns if c.startswith("obj:")]
print("\nObjective columns (use as OBJECTIVES_BOUNDS keys):")
for c in _obj_cols:
    print(f"  {c!r}")


2026-10-05 16:20:29,377 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:29,378 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.
2026-10-05 16:20:29,378 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:29,378 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


Artifacts downloaded to: .temp/Obj

Objective columns (use as OBJECTIVES_BOUNDS keys):
  'obj:max(μ(product_concentration))'
  'obj:max(σ(product_concentration))'


In [73]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_full = update_pareto_front_plots(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    clicked_points={},
    objectives_bounds=OBJECTIVES_BOUNDS,
    pinned_points={},
    objective_renames=OBJECTIVE_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    measurement_renames=MEASUREMENT_RENAMES,
    pareto_mode=PARETO_MODE,
)

# ── Row 1 only ────────────────────────────────────────────────────────────────
layout_dict = fig_full.to_plotly_json()["layout"]

# Row 1 yaxes: domain[1] ≈ 1.0 (topmost row always reaches the top)
row1_y_refs = {
    "y" + key[5:]
    for key, val in layout_dict.items()
    if re.match(r"^yaxis\d*$", key)
    and isinstance(val, dict)
    and abs((val.get("domain") or [0, 0])[1] - 1.0) < 0.01
}
row1_x_refs = {"x" + ref[1:] for ref in row1_y_refs}

# Map axis ref → subplot column (sorted: 'x'→1, 'x2'→2, 'x3'→3)
def _col_num(ref: str) -> int:
    n = ref[1:]  # strip leading 'x'
    return int(n) if n else 1

sorted_x_refs = sorted(row1_x_refs, key=_col_num)
x_ref_to_col = {ref: i + 1 for i, ref in enumerate(sorted_x_refs)}
n_cols = len(sorted_x_refs)

# Build proper 1-row subplot figure
fig = make_subplots(rows=1, cols=n_cols, horizontal_spacing=0.12)
fig.update_layout(template=fig_full.layout.template)

# Add row-1 traces with explicit col so xaxis/yaxis are always set correctly
for trace in fig_full.data:
    x_ref = trace.xaxis or "x"
    y_ref = trace.yaxis or "y"
    if y_ref in row1_y_refs:
        col = x_ref_to_col.get(x_ref, 1)
        fig.add_trace(trace, row=1, col=col)

# Copy axis titles/ticks from original (skip domain; make_subplots handles it)
for key, val in layout_dict.items():
    if not isinstance(val, dict):
        continue
    if re.match(r"^xaxis\d*$", key) and "x" + key[5:] in row1_x_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})
    elif re.match(r"^yaxis\d*$", key) and "y" + key[5:] in row1_y_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})

# Copy row-1 shapes (bounds vlines/hlines) using subplot-aware placement
for shape in fig_full.layout.shapes:
    s = shape.to_plotly_json()
    xref_base = (s.get("xref") or "").split(" ")[0]
    if xref_base not in row1_x_refs:
        continue
    col = x_ref_to_col.get(xref_base, 1)
    line = s.get("line", {})
    x0, x1 = s.get("x0"), s.get("x1")
    y0, y1 = s.get("y0"), s.get("y1")
    if x0 is not None and x0 == x1:          # vertical bound line
        fig.add_vline(x=float(x0), row=1, col=col, line=line)
    elif y0 is not None and y0 == y1:         # horizontal bound line
        fig.add_hline(y=float(y0), row=1, col=col, line=line)
    else:                                     # diagonal or other shape
        fig.add_shape(
            **{k: v for k, v in s.items() if k not in ("xref", "yref")},
            row=1, col=col,
        )

fig.update_layout(
    showlegend=layout_dict.get("showlegend", True),
    hovermode=layout_dict.get("hovermode"),
    legend=layout_dict.get("legend"),
    width=layout_dict.get("width"),
    height=500,
)

# ── Diagonal gray dashed line on second subplot (col 2) ───────────────────────
x2_vals, y2_vals = [], []
for trace in fig.data:
    if trace.xaxis == "x2" and trace.yaxis == "y2":
        if getattr(trace, "x", None) is not None:
            x2_vals.extend(v for v in trace.x if v is not None)
        if getattr(trace, "y", None) is not None:
            y2_vals.extend(v for v in trace.y if v is not None)

if x2_vals and y2_vals:
    diag_min = min(min(x2_vals), min(y2_vals))
    diag_max = max(max(x2_vals), max(y2_vals))
    fig.add_shape(
        type="line",
        x0=diag_min, y0=diag_min,
        x1=diag_max, y1=diag_max,
        row=1, col=2,
        line=dict(color="gray", width=2, dash="dash"),
    )

fig.update_layout(showlegend=True, margin=dict(l=100, r=20, t=130, b=100), width=600, height=600,legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99, font=dict(size=30)))
for col_idx, (x_name, y_name) in enumerate(zip(x_axis_names, y_axis_names), start=1):
    fig.update_xaxes(title_text=x_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30),)
    fig.update_yaxes(title_text=y_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30))
fig.update_layout(title=dict(text="Pareto Front (Iteration 2)", font=dict(size=22), x=0.5, xanchor="center"))
fig.show()
fig_f = fig
fig.write_image(f"{PDF_FOLDER}/it1_ms_pareto-front.pdf")

2026-10-05 16:20:29,428 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:29,429 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:29,430 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpzzikt5jw.
2026-10-05 16:20:29,431 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:29,431 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpvxe_61m3.
2026-10-05 16:20:29,432 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpvxe_61m3
2026-10-05 16:20:29,841 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpzzikt5jw/index.html
2026-10-05 16:20:30,616 - kaleido.kaleido - INFO - Getting tab from queue 

## GP Prediction: Product Concentration

In [74]:
from paretodo.dashboard.callbacks.gp_prediction import store_gp_predictions_figures
from paretodo.data_models import InvestigationSpace
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# Load default filter values: first (sorted) value per variable in the meshgrid
_meshgrid_df = pd.read_csv(f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/{ITERATION}/meshgrid_predictions.csv")
_inv_space = InvestigationSpace.from_json(
    Path(f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/general/investigation_space.json")
)
_var_names = [v.name for v in _inv_space.variables]
_filter_values = [float(sorted(_meshgrid_df[v].unique())[0]) for v in _var_names]

ITERATION = "Iteration_001"
INVESTIGATION_SPACE_RENAMES = {"temperature": "Temperature [°C]", "ph_value": "pH [-]"}
OBJECTIVES_BOUNDS = {}
gp_figs = store_gp_predictions_figures(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    filter_values=_filter_values,
    measurement_renames=MEASUREMENT_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    objectives_bounds=OBJECTIVES_BOUNDS,
    robustness_confidence_level=None,
    robustness_pareto_point=None,
    alternative_expected_errors=None,
)

GP_MATCH = "product_concentration"
for fig_dict in gp_figs:
    title = fig_dict.get("layout", {}).get("title", {}).get("text", "")
    if GP_MATCH not in title:
        continue
    fig = go.Figure(fig_dict)
    fig.data[0].showlegend = False
    fig.data[1].showlegend = True
    fig.data[1].name = "Experiments"
    fig.update_layout(width=1000, height=700, margin=dict(l=100, r=20, t=90, b=110), legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.2, font=dict(size=30)), title=dict(text="GP Prediction: Titer (Iteration 1)", font=dict(size=26), x=0.5, xanchor="center"), coloraxis=dict(cauto=False, cmin=200, cmax=2200), )
    fig.update_xaxes(title_font_size=30, tickfont=dict(size=30))
    fig.update_yaxes(title_font_size=30, tickfont=dict(size=30))
    fig.show()
    fig.write_image(f"{PDF_FOLDER}/init_ms_gp-predictions-01.pdf")
fig_a = fig


2026-10-05 16:20:30,896 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:30,897 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-10-05 16:20:30,919 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:30,919 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:30,920 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpgt46xy79.
2026-10-05 16:20:30,921 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:30,921 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpuk9uxk1d.
2026-10-05 16:20:30,921 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpuk9uxk1d
2026-10-05 16:20:31,368 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpgt46xy79/index.html
2026-10-05 16:20:31,941 - kaleido.kaleido - INFO - Getting tab from queue 

In [75]:
from paretodo.dashboard.callbacks.gp_prediction import store_gp_predictions_figures
from paretodo.data_models import InvestigationSpace
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# Load default filter values: first (sorted) value per variable in the meshgrid
_meshgrid_df = pd.read_csv(f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/{ITERATION}/meshgrid_predictions.csv")
_inv_space = InvestigationSpace.from_json(
    Path(f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/general/investigation_space.json")
)
_var_names = [v.name for v in _inv_space.variables]
_filter_values = [float(sorted(_meshgrid_df[v].unique())[0]) for v in _var_names]

ITERATION = "Iteration_001"
INVESTIGATION_SPACE_RENAMES = {"temperature": "Temperature [°C]", "ph_value": "pH [-]"}
OBJECTIVES_BOUNDS = {
    "obj:max(σ(product_concentration))": {"min": 250, "max": None},
    "obj:max(μ(product_concentration))": {"min": 1600,   "max": None},
}
gp_figs = store_gp_predictions_figures(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    filter_values=_filter_values,
    measurement_renames=MEASUREMENT_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    objectives_bounds=OBJECTIVES_BOUNDS,
    robustness_confidence_level=None,
    robustness_pareto_point=None,
    alternative_expected_errors=None,
)

GP_MATCH = "product_concentration"
for fig_dict in gp_figs:
    title = fig_dict.get("layout", {}).get("title", {}).get("text", "")
    if GP_MATCH not in title:
        continue
    fig = go.Figure(fig_dict)
    fig.data[0].showlegend = False
    fig.data[1].showlegend = True
    fig.data[1].name = "Experiments"
    fig.update_layout(width=1000, height=700, margin=dict(l=100, r=20, t=90, b=110),coloraxis=dict(cauto=False, cmin=200, cmax=2200), legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.2, font=dict(size=30)), title=dict(text="GP Prediction: Titer (Expert Limited & Selected Experiments)", font=dict(size=26), x=0.5, xanchor="center"))
    fig.update_xaxes(title_font_size=30, tickfont=dict(size=30))
    fig.update_yaxes(title_font_size=30, tickfont=dict(size=30))
    fig.show()
    fig.write_image(f"{PDF_FOLDER}/init_ms_rest_gp-predictions-01.pdf")
fig_c = fig

2026-10-05 16:20:32,240 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:32,241 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-10-05 16:20:32,271 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:32,272 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:32,273 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp5nqz_8_u.
2026-10-05 16:20:32,274 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:32,274 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmppqmag1x9.
2026-10-05 16:20:32,275 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmppqmag1x9
2026-10-05 16:20:32,691 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp5nqz_8_u/index.html
2026-10-05 16:20:33,264 - kaleido.kaleido - INFO - Getting tab from queue 

In [76]:
from paretodo.dashboard.callbacks.gp_prediction import store_gp_predictions_figures
from paretodo.data_models import InvestigationSpace
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# Load default filter values: first (sorted) value per variable in the meshgrid
_meshgrid_df = pd.read_csv(f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/{ITERATION}/meshgrid_predictions.csv")
_inv_space = InvestigationSpace.from_json(
    Path(f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/general/investigation_space.json")
)
_var_names = [v.name for v in _inv_space.variables]
_filter_values = [float(sorted(_meshgrid_df[v].unique())[0]) for v in _var_names]

ITERATION = "Iteration_002"
INVESTIGATION_SPACE_RENAMES = {"temperature": "Temperature [°C]", "ph_value": "pH [-]"}
OBJECTIVES_BOUNDS = {}
gp_figs = store_gp_predictions_figures(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    filter_values=_filter_values,
    measurement_renames=MEASUREMENT_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    objectives_bounds=OBJECTIVES_BOUNDS,
    robustness_confidence_level=None,
    robustness_pareto_point=None,
    alternative_expected_errors=None,
)

GP_MATCH = "product_concentration"
for fig_dict in gp_figs:
    title = fig_dict.get("layout", {}).get("title", {}).get("text", "")
    if GP_MATCH not in title:
        continue
    fig = go.Figure(fig_dict)
    fig.data[0].showlegend = False
    fig.data[1].showlegend = True
    fig.data[1].name = "Experiments"
    fig.update_layout(width=1000, height=700, margin=dict(l=100, r=20, t=90, b=110),coloraxis=dict(cauto=False, cmin=200, cmax=2200), legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.2, font=dict(size=30)), title=dict(text="GP Prediction of Titer (Iteration 2)", font=dict(size=26), x=0.5, xanchor="center"))
    fig.update_xaxes(title_font_size=30, tickfont=dict(size=30))
    fig.update_yaxes(title_font_size=30, tickfont=dict(size=30))
    fig.show()
    fig.write_image(f"{PDF_FOLDER}/ti1_ms_gp-predictions-01.pdf")
fig_e = fig

2026-10-05 16:20:33,574 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:33,575 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-10-05 16:20:33,595 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:33,596 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:33,596 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmprahsvz0y.
2026-10-05 16:20:33,597 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:33,598 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp1iy75vjt.
2026-10-05 16:20:33,598 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp1iy75vjt
2026-10-05 16:20:34,017 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmprahsvz0y/index.html
2026-10-05 16:20:34,602 - kaleido.kaleido - INFO - Getting tab from queue 

In [77]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

panel_figs = [
    (1, 1, fig_a),
    (1, 2, fig_b),
    (2, 1, fig_c),
    (2, 2, fig_d),
    (3, 1, fig_e),
    (3, 2, fig_f),
]

def _combined_suffix(row, col, n_cols=2):
    idx = (row - 1) * n_cols + col
    return "" if idx == 1 else str(idx)

def _remap_coloraxis(td, ca_remap):
    if not ca_remap:
        return td
    td = dict(td)
    if td.get("coloraxis") in ca_remap:
        td["coloraxis"] = ca_remap[td["coloraxis"]]
    if "marker" in td and isinstance(td["marker"], dict):
        m = dict(td["marker"])
        if m.get("coloraxis") in ca_remap:
            m["coloraxis"] = ca_remap[m["coloraxis"]]
        td["marker"] = m
    return td

def _is_orange_diamond(td):
    m = td.get("marker", {})
    return (isinstance(m, dict)
            and m.get("symbol") == "diamond"
            and m.get("color") == "#FFA500")

GP_ZMIN, GP_ZMAX = 200, 2200
GP_CA = "coloraxis"

subplot_figure = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=["GP Visualization", "Pareto Front"] * 3,
    horizontal_spacing=0.15,
    vertical_spacing=0.15,
 )
subplot_figure.update_layout(template=fig_a.layout.template)

seen_names = set()
axis_struct_keys = {"domain", "anchor", "overlaying", "side", "position"}

gp_ca_setup = False
pareto_ca_counter = 1

for row, col, fig in panel_figs:
    layout_dict = fig.to_plotly_json()["layout"]
    sfx = _combined_suffix(row, col)
    
    # --- coloraxis setup ---
    ca_remap = {}
    if col == 1:
        # Build the shared GP coloraxis on first GP panel.
        # Source figures may use inline colorscale (no coloraxis key in layout),
        # so we fall back to extracting the colorscale from the first z-trace.
        if not gp_ca_setup:
            ca_props = {}
            src_cas = sorted(k for k in layout_dict if re.match(r"^coloraxis\d*$", k))
            if src_cas:
                ca_props = {k: v for k, v in layout_dict[src_cas[0]].items()
                            if k not in ("cmin", "cmax", "cauto")}
            else:
                for t in fig.data:
                    td0 = t.to_plotly_json()
                    if "z" in td0 and td0.get("colorscale"):
                        ca_props = {"colorscale": td0["colorscale"]}
                        break
            ca_props.update({"cauto": False, "cmin": GP_ZMIN, "cmax": GP_ZMAX,
                             "showscale": True,
                             "colorbar": {"title": {"text": "Titer [mg/L]"}}})
            subplot_figure.update_layout({GP_CA: ca_props})
            gp_ca_setup = True
        # remap any explicit coloraxis refs in the source layout
        for src_ca in sorted(k for k in layout_dict if re.match(r"^coloraxis\d*$", k)):
            ca_remap[src_ca] = GP_CA
    else:
        for src_ca in sorted(k for k in layout_dict if re.match(r"^coloraxis\d*$", k)):
            pareto_ca_counter += 1
            new_ca = f"coloraxis{pareto_ca_counter}"
            ca_remap[src_ca] = new_ca
            subplot_figure.update_layout({new_ca: layout_dict[src_ca]})

    # --- traces ---
    for trace in fig.data:
        td = _remap_coloraxis(trace.to_plotly_json(), ca_remap)
        if _is_orange_diamond(td):
            td = {**td, "name": "Selected Recipes"}

        # Force every z-trace in a GP panel onto the shared coloraxis,
        # removing the per-trace colorbar so only one colorbar is shown.
        if col == 1 and "z" in td:
            td = {k: v for k, v in td.items() if k not in ("showscale", "colorscale", "colorbar")}
            td["coloraxis"] = GP_CA

        name = td.get("name")
        show = td.get("showlegend", True)
        if name and show:
            if name in seen_names:
                td = {**td, "showlegend": False}
            else:
                seen_names.add(name)
        subplot_figure.add_trace(go.Figure({"data": [td]}).data[0], row=row, col=col)

    # --- axis properties ---
    for key, val in layout_dict.items():
        if not isinstance(val, dict):
            continue
        if key == "xaxis":
            subplot_figure.update_layout({"xaxis" + sfx: {k: v for k, v in val.items() if k not in axis_struct_keys}})
        elif key == "yaxis":
            subplot_figure.update_layout({"yaxis" + sfx: {k: v for k, v in val.items() if k not in axis_struct_keys}})

    # --- shapes ---
    for shape in fig.layout.shapes:
        s = shape.to_plotly_json()
        x0, x1 = s.get("x0"), s.get("x1")
        y0, y1 = s.get("y0"), s.get("y1")
        line = s.get("line", {})
        if x0 is not None and x0 == x1:
            subplot_figure.add_vline(x=float(x0), row=row, col=col, line=line)
        elif y0 is not None and y0 == y1:
            subplot_figure.add_hline(y=float(y0), row=row, col=col, line=line)
        else:
            subplot_figure.add_shape(**{k: v for k, v in s.items() if k not in ("xref", "yref")}, row=row, col=col)

# Hide all non-GP colorbars
for ca_key in sorted(k for k in subplot_figure.layout.to_plotly_json() if re.match(r"^coloraxis\d*$", k)):
    if ca_key != GP_CA:
        subplot_figure.update_layout({ca_key: {"showscale": False}})

# Orange diamonds on panel (c) in parameter space
try:
    _par_cols = [c for c in pareto_df.columns if not c.startswith("obj:")]
    _rev = {v: k for k, v in INVESTIGATION_SPACE_RENAMES.items()}
    _x_col = _rev.get((fig_c.layout.xaxis.title.text or "").strip(), _par_cols[0] if _par_cols else None)
    _y_col = _rev.get((fig_c.layout.yaxis.title.text or "").strip(), _par_cols[1] if len(_par_cols) > 1 else None)
    _sel = pareto_df.iloc[EXPERIMENT_RECIPES]
    if _x_col and _y_col and _x_col in _sel.columns and _y_col in _sel.columns:
        subplot_figure.add_trace(
            go.Scatter(
                x=_sel[_x_col], y=_sel[_y_col],
                mode="markers",
                marker=dict(size=18, symbol="diamond", color="#FFA500",
                            line=dict(width=2, color="black")),
                name="Selected Recipes",
                legendgroup="selected_recipes",
                showlegend=False,
            ),
            row=2, col=1,
        )
    else:
        print(f"Column mismatch — x={_x_col!r}, y={_y_col!r}, par_cols={_par_cols}")
except Exception as e:
    print(f"Could not add selected recipes to panel (c): {e}")


# Column headers come from subplot_titles (row 1 only) — just enlarge them
annotations = list(subplot_figure.layout.annotations)
for annotation in annotations:
    annotation.font.size = 30

# Panel labels centered below each x-axis title
labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]
for label, (row, col, _) in zip(labels, panel_figs):
    sfx = _combined_suffix(row, col)
    annotations.append(dict(
        text=f"<b>{label}</b>",
        x=0.5, y=-0.38,
        xref="x" + sfx + " domain",
        yref="y" + sfx + " domain",
        xanchor="center", yanchor="top",
        showarrow=False,
        font=dict(size=22),
    ))

# Row labels on the left: per-row experiment set, plus a bracketed iteration group
ROW_DESCRIPTIONS = [
    "Possible Experiments",
    "Expert Limited &<br>Selected Experiments",
    "Possible Experiments",
]
ITERATION_GROUPS = [("Iteration 1", 1, 2), ("Iteration 2", 3, 3)]
DESC_X, BRACKET_X, GROUP_X = -0.17, -0.21, -0.24

def _row_domain(row):
    domain = subplot_figure.layout["yaxis" + _combined_suffix(row, 1)].domain
    return float(domain[0]), float(domain[1])

for row, description in enumerate(ROW_DESCRIPTIONS, start=1):
    y0, y1 = _row_domain(row)
    annotations.append(dict(
        text=description,
        x=DESC_X, y=(y0 + y1) / 2,
        xref="paper", yref="paper",
        textangle=-90, xanchor="center", yanchor="middle",
        showarrow=False,
        font=dict(size=24),
    ))

for name, first_row, last_row in ITERATION_GROUPS:
    y_top = _row_domain(first_row)[1]
    y_bottom = _row_domain(last_row)[0]
    annotations.append(dict(
        text=f"<b>{name}</b>",
        x=GROUP_X, y=(y_top + y_bottom) / 2,
        xref="paper", yref="paper",
        textangle=-90, xanchor="center", yanchor="middle",
        showarrow=False,
        font=dict(size=30),
    ))
    subplot_figure.add_shape(
        type="line",
        xref="paper", yref="paper",
        x0=BRACKET_X, x1=BRACKET_X, y0=y_bottom, y1=y_top,
        line=dict(color="black", width=2),
    )

subplot_figure.update_layout(
    width=1800,
    height=1800,
    showlegend=True,
    legend=dict(yanchor="top", y=-0.15, xanchor="center", x=0.5, font=dict(size=30), orientation="h",entrywidth=500 ),
    margin=dict(l=340, r=50, t=100, b=150),
    annotations=annotations,
    coloraxis=dict(colorbar=dict(
        title=dict(text="Titer [mg/L]", font=dict(size=30)),
        tickfont=dict(size=30, color="black"),
        ticks="outside",
        ticklen=10,
        tickwidth=2,
        tickcolor="black",
        dtick=400,
        thickness=30,
        outlinewidth=1,
        outlinecolor="black",
    )),
)
subplot_figure.show()
subplot_figure.write_image(f"{PDF_FOLDER}/pfgs_obj.pdf")

2026-10-05 16:20:34,878 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:34,878 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-10-05 16:20:34,933 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:34,933 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:34,934 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpvkatlndj.
2026-10-05 16:20:34,935 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:34,935 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp11nqs69h.
2026-10-05 16:20:34,936 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp11nqs69h
2026-10-05 16:20:35,364 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpvkatlndj/index.html
2026-10-05 16:20:35,951 - kaleido.kaleido - INFO - Getting tab from queue 

# Constraints

In [78]:
# MLflow tracking URI
# Local SQLite:  "sqlite:////absolute/path/to/mlflow.db"
# Running server: "http://127.0.0.1:5000"
MLFLOW_TRACKING_URI = "sqlite:////Users/ss15425/Documents/00_Git_Repositories/pfgs-research/mlflow.db"

MLFLOW_EXPERIMENT = "Journal-02-06-2026"
MLFLOW_RUN = "Constr_Obj"
ITERATION = "Iteration_001"

# Renames: map internal column names to display labels (leave empty for defaults)
OBJECTIVE_RENAMES: dict[str, str] = {}
INVESTIGATION_SPACE_RENAMES: dict[str, str] = {}
MEASUREMENT_RENAMES: dict[str, str] = {}

# Bounds — keys must match the exact objective column names printed by cell 3.
# Points outside ANY bound appear as black "Discarded" markers and red dashed lines.
# Uncomment and fill in the bounds you want:
OBJECTIVES_BOUNDS = {
    "obj:max(σ(product_concentration))": {"min": 100, "max": None},
    "obj:max(μ(product_concentration))": {"min": 1300,   "max": None},
    "obj:max(P(purity))": {"min": 0.8,   "max": None},
}
# OBJECTIVES_BOUNDS: dict[str, dict] = {}

PARETO_MODE = "Optimization"  # "Optimization" or "Manufacturing"

x_axis_names = ["μ(Titer) [mg/L]", "μ(Titer) [mg/L]", "μ(Titer) [mg/L]"]
y_axis_names = ["σ(Titer) [mg/L]", "P(Purity > 0.8) [-]", "P(Purity > 0.8) [-]"]
subplot_titles = ["Mean Prediction vs. Standard Deviation", "Mean Prediction vs. Constraint Probability", "Mean Prediction vs. Constraint Probability"]


In [79]:
import pandas as pd

client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
runs = client.search_runs(experiment_ids=[experiment.experiment_id])
selected_run = next(r for r in runs if r.info.run_name == MLFLOW_RUN)
run_id = selected_run.info.run_id

temp_dir = os.path.join(LOCAL_TEMP_DIR, MLFLOW_RUN)
os.makedirs(temp_dir, exist_ok=True)

client.download_artifacts(run_id=run_id, path="general", dst_path=temp_dir)
client.download_artifacts(run_id=run_id, path=ITERATION, dst_path=temp_dir)

print(f"Artifacts downloaded to: {temp_dir}")

# Show objective column names — use these as keys in OBJECTIVES_BOUNDS
_pareto_csv = f"{temp_dir}/{ITERATION}/pareto_front_predictions.csv"
_obj_cols = [c for c in pd.read_csv(_pareto_csv, nrows=0).columns if c.startswith("obj:")]
print("\nObjective columns (use as OBJECTIVES_BOUNDS keys):")
for c in _obj_cols:
    print(f"  {c!r}")


2026-10-05 16:20:36,390 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:36,390 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.

Artifacts downloaded to: .temp/Constr_Obj

Objective columns (use as OBJECTIVES_BOUNDS keys):
  'obj:max(μ(product_concentration))'
  'obj:max(σ(product_concentration))'
  'obj:max(P(purity))'


In [80]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_full = update_pareto_front_plots(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    clicked_points={},
    objectives_bounds=OBJECTIVES_BOUNDS,
    pinned_points={},
    objective_renames=OBJECTIVE_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    measurement_renames=MEASUREMENT_RENAMES,
    pareto_mode=PARETO_MODE,
)

# ── Row 1 only ────────────────────────────────────────────────────────────────
layout_dict = fig_full.to_plotly_json()["layout"]

# Row 1 yaxes: domain[1] ≈ 1.0 (topmost row always reaches the top)
row1_y_refs = {
    "y" + key[5:]
    for key, val in layout_dict.items()
    if re.match(r"^yaxis\d*$", key)
    and isinstance(val, dict)
    and abs((val.get("domain") or [0, 0])[1] - 1.0) < 0.01
}
row1_x_refs = {"x" + ref[1:] for ref in row1_y_refs}

# Map axis ref → subplot column (sorted: 'x'→1, 'x2'→2, 'x3'→3)
def _col_num(ref: str) -> int:
    n = ref[1:]  # strip leading 'x'
    return int(n) if n else 1

sorted_x_refs = sorted(row1_x_refs, key=_col_num)
x_ref_to_col = {ref: i + 1 for i, ref in enumerate(sorted_x_refs)}
n_cols = 3

# Build proper 1-row subplot figure
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12, subplot_titles=subplot_titles[:n_cols])
fig.update_layout(template=fig_full.layout.template)

# Add row-1 traces with explicit col so xaxis/yaxis are always set correctly
for trace in fig_full.data:
    x_ref = trace.xaxis or "x"
    y_ref = trace.yaxis or "y"
    if y_ref in row1_y_refs:
        col = x_ref_to_col.get(x_ref, 1)
        if col >= n_cols:
            continue
        fig.add_trace(trace, row=1, col=col)

# Copy axis titles/ticks from original (skip domain; make_subplots handles it)
for key, val in layout_dict.items():
    if not isinstance(val, dict):
        continue
    if re.match(r"^xaxis\d*$", key) and "x" + key[5:] in row1_x_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})
    elif re.match(r"^yaxis\d*$", key) and "y" + key[5:] in row1_y_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})

# Copy row-1 shapes (bounds vlines/hlines) using subplot-aware placement
for shape in fig_full.layout.shapes:
    s = shape.to_plotly_json()
    xref_base = (s.get("xref") or "").split(" ")[0]
    if xref_base not in row1_x_refs:
        continue
    col = x_ref_to_col.get(xref_base, 1)
    line = s.get("line", {})
    x0, x1 = s.get("x0"), s.get("x1")
    y0, y1 = s.get("y0"), s.get("y1")
    if x0 is not None and x0 == x1:          # vertical bound line
        fig.add_vline(x=float(x0), row=1, col=col, line=line)
    elif y0 is not None and y0 == y1:         # horizontal bound line
        fig.add_hline(y=float(y0), row=1, col=col, line=line)
    else:                                     # diagonal or other shape
        fig.add_shape(
            **{k: v for k, v in s.items() if k not in ("xref", "yref")},
            row=1, col=col,
        )

fig.update_layout(
    showlegend=layout_dict.get("showlegend", True),
    hovermode=layout_dict.get("hovermode"),
    legend=layout_dict.get("legend"),
    width=layout_dict.get("width"),
    height=500,
)

# # ── Diagonal gray dashed line on second subplot (col 2) ───────────────────────
# x2_vals, y2_vals = [], []
# for trace in fig.data:
#     if trace.xaxis == "x2" and trace.yaxis == "y2":
#         if getattr(trace, "x", None) is not None:
#             x2_vals.extend(v for v in trace.x if v is not None)
#         if getattr(trace, "y", None) is not None:
#             y2_vals.extend(v for v in trace.y if v is not None)

# if x2_vals and y2_vals:
#     diag_min = min(min(x2_vals), min(y2_vals))
#     diag_max = max(max(x2_vals), max(y2_vals))
#     fig.add_shape(
#         type="line",
#         x0=diag_min, y0=diag_min,
#         x1=diag_max, y1=diag_max,
#         row=1, col=2,
#         line=dict(color="gray", width=2, dash="dash"),
#     )

fig.update_layout(showlegend=True, margin=dict(l=100, r=20, t=110, b=120), width=2000, height=600,legend=dict(yanchor="bottom", y=0.01, xanchor="left", x=0.375, font=dict(size=30)))
for col_idx, (x_name, y_name) in enumerate(zip(x_axis_names, y_axis_names), start=1):
    fig.update_xaxes(title_text=x_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30),)
    fig.update_yaxes(title_text=y_name, row=1, col=col_idx, title_font_size=30, tickfont=dict(size=30))
for annotation in fig.layout.annotations:
    annotation.font.size = 25

# Iteration label on the left, spanning the whole row
# fig.add_annotation(
#     text=f"<b>Iteration {int(ITERATION.split('_')[-1])}</b>",
#     x=-0.09, y=0.5,
#     xref="paper", yref="paper",
#     textangle=-90, xanchor="center", yanchor="middle",
#     showarrow=False,
#     font=dict(size=32),
# )
fig.show()
fig.write_image(f"{PDF_FOLDER}/initial_msp_res_pareto-front.pdf")


2026-10-05 16:20:36,476 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:36,477 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:36,477 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp3lq035ol.
2026-10-05 16:20:36,478 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:36,479 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpx82zdlyc.
2026-10-05 16:20:36,479 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpx82zdlyc
2026-10-05 16:20:36,911 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp3lq035ol/index.html
2026-10-05 16:20:37,517 - kaleido.kaleido - INFO - Getting tab from queue 

# Robstness

In [81]:
# MLflow tracking URI
# Local SQLite:  "sqlite:////absolute/path/to/mlflow.db"
# Running server: "http://127.0.0.1:5000"
MLFLOW_TRACKING_URI = "sqlite:////Users/ss15425/Documents/00_Git_Repositories/pfgs-research/mlflow.db"

MLFLOW_EXPERIMENT = "Journal-02-06-2026"
MLFLOW_RUN = "Robust_Obj"
ITERATION = "Iteration_005"

# Renames: map internal column names to display labels (leave empty for defaults)
OBJECTIVE_RENAMES: dict[str, str] = {}
INVESTIGATION_SPACE_RENAMES: dict[str, str] = {}
MEASUREMENT_RENAMES: dict[str, str] = {}

# Bounds — keys must match the exact objective column names printed by cell 3.
# Points outside ANY bound appear as black "Discarded" markers and red dashed lines.
# Uncomment and fill in the bounds you want:
OBJECTIVES_BOUNDS = {
    # "obj:max(σ(product_concentration))": {"min": 100, "max": None},
    # "obj:max(μ(product_concentration))": {"min": 1300,   "max": None},
    # "obj:max(P(purity))": {"min": 0.8,   "max": None},
}
# OBJECTIVES_BOUNDS: dict[str, dict] = {}

PARETO_MODE = "Optimization"  # "Optimization" or "Manufacturing"

x_axis_names = ["μ(Titer) [mg/L]", "μ(Titer) [mg/L]", "μ(Titer) [mg/L]"]
y_axis_names = ["σ(Titer) [mg/L]", "Robustness (Titer) [mg/L]", "Robustness (Titer) [mg/L]"]
subplot_titles = ["Mean Prediction vs. Standard Deviation", "Mean Prediction vs. Robustness", "Mean Prediction vs. Robustness"]


In [82]:
import pandas as pd

client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
runs = client.search_runs(experiment_ids=[experiment.experiment_id])
selected_run = next(r for r in runs if r.info.run_name == MLFLOW_RUN)
run_id = selected_run.info.run_id

temp_dir = os.path.join(LOCAL_TEMP_DIR, MLFLOW_RUN)
os.makedirs(temp_dir, exist_ok=True)

client.download_artifacts(run_id=run_id, path="general", dst_path=temp_dir)
client.download_artifacts(run_id=run_id, path=ITERATION, dst_path=temp_dir)

print(f"Artifacts downloaded to: {temp_dir}")

# Show objective column names — use these as keys in OBJECTIVES_BOUNDS
_pareto_csv = f"{temp_dir}/{ITERATION}/pareto_front_predictions.csv"
_obj_cols = [c for c in pd.read_csv(_pareto_csv, nrows=0).columns if c.startswith("obj:")]
print("\nObjective columns (use as OBJECTIVES_BOUNDS keys):")
for c in _obj_cols:
    print(f"  {c!r}")


2026-10-05 16:20:37,846 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:37,846 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.

Artifacts downloaded to: .temp/Robust_Obj

Objective columns (use as OBJECTIVES_BOUNDS keys):
  'obj:max(μ(product_concentration))'
  'obj:max(σ(product_concentration))'
  'obj:max(R(product_concentration))'


In [ ]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_full = update_pareto_front_plots(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    clicked_points={},
    objectives_bounds=OBJECTIVES_BOUNDS,
    pinned_points={},
    objective_renames=OBJECTIVE_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    measurement_renames=MEASUREMENT_RENAMES,
    pareto_mode=PARETO_MODE,
)

# ── Row 1 only ────────────────────────────────────────────────────────────────
layout_dict = fig_full.to_plotly_json()["layout"]

# Row 1 yaxes: domain[1] ≈ 1.0 (topmost row always reaches the top)
row1_y_refs = {
    "y" + key[5:]
    for key, val in layout_dict.items()
    if re.match(r"^yaxis\d*$", key)
    and isinstance(val, dict)
    and abs((val.get("domain") or [0, 0])[1] - 1.0) < 0.01
}
row1_x_refs = {"x" + ref[1:] for ref in row1_y_refs}

# Map axis ref → subplot column (sorted: 'x'→1, 'x2'→2, 'x3'→3)
def _col_num(ref: str) -> int:
    n = ref[1:]  # strip leading 'x'
    return int(n) if n else 1

sorted_x_refs = sorted(row1_x_refs, key=_col_num)
x_ref_to_col = {ref: i + 1 for i, ref in enumerate(sorted_x_refs)}
n_cols = len(sorted_x_refs)

# Build proper 1-row subplot figßre
fig = make_subplots(rows=1, cols=n_cols, horizontal_spacing=0.12, subplot_titles=subplot_titles[:n_cols])
fig.update_layout(template=fig_full.layout.template)

# Add row-1 traces with explicit col so xaxis/yaxis are always set correctly
for trace in fig_full.data:
    x_ref = trace.xaxis or "x"
    y_ref = trace.yaxis or "y"
    if y_ref in row1_y_refs:
        col = x_ref_to_col.get(x_ref, 1)
        fig.add_trace(trace, row=1, col=col)

# Copy axis titles/ticks from original (skip domain; make_subplots handles it)
for key, val in layout_dict.items():
    if not isinstance(val, dict):
        continue
    if re.match(r"^xaxis\d*$", key) and "x" + key[5:] in row1_x_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})
    elif re.match(r"^yaxis\d*$", key) and "y" + key[5:] in row1_y_refs:
        fig.update_layout({key: {k: v for k, v in val.items() if k != "domain"}})

# Copy row-1 shapes (bounds vlines/hlines) using subplot-aware placement
for shape in fig_full.layout.shapes:
    s = shape.to_plotly_json()
    xref_base = (s.get("xref") or "").split(" ")[0]
    if xref_base not in row1_x_refs:
        continue
    col = x_ref_to_col.get(xref_base, 1)
    line = s.get("line", {})
    x0, x1 = s.get("x0"), s.get("x1")
    y0, y1 = s.get("y0"), s.get("y1")
    if x0 is not None and x0 == x1:          # vertical bound line
        fig.add_vline(x=float(x0), row=1, col=col, line=line)
    elif y0 is not None and y0 == y1:         # horizontal bound line
        fig.add_hline(y=float(y0), row=1, col=col, line=line)
    else:                                     # diagonal or other shape
        fig.add_shape(
            **{k: v for k, v in s.items() if k not in ("xref", "yref")},
            row=1, col=col,
        )

fig.update_layout(
    showlegend=layout_dict.get("showlegend", True),
    hovermode=layout_dict.get("hovermode"),
    legend=layout_dict.get("legend"),
    width=layout_dict.get("width"),
    height=500,
)

# ── Diagonal gray dashed line on second subplot (col 2) ───────────────────────
x2_vals, y2_vals = [], []
for trace in fig.data:
    if trace.xaxis == "x2" and trace.yaxis == "y2":
        if getattr(trace, "x", None) is not None:
            x2_vals.extend(v for v in trace.x if v is not None)
        if getattr(trace, "y", None) is not None:
            y2_vals.extend(v for v in trace.y if v is not None)

if x2_vals and y2_vals:
    diag_min = min(min(x2_vals), min(y2_vals))
    diag_max = max(max(x2_vals), max(y2_vals))
    fig.add_shape(
        type="line",
        x0=diag_min, y0=diag_min,
        x1=diag_max, y1=diag_max,
        row=1, col=2,
        line=dict(color="gray", width=2, dash="dash"),
    )

fig.update_layout(showlegend=True, margin=dict(l=240, r=20, t=110, b=100), width=2000, height=600,legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.375, font=dict(size=30)))
for col_idx, (x_name, y_name) in enumerate(zip(x_axis_names, y_axis_names), start=1):
    fig.update_xaxes(title_text=x_name, row=1, col=col_idx, title_font_size=20, tickfont=dict(size=20),)
    fig.update_yaxes(title_text=y_name, row=1, col=col_idx, title_font_size=20, tickfont=dict(size=20))
for annotation in fig.layout.annotations:
    annotation.font.size = 25

# Iteration label on the left, spanning the whole row
fig.add_annotation(
    text=f"<b>Iteration {int(ITERATION.split('_')[-1])}</b>",
    x=-0.09, y=0.5,
    xref="paper", yref="paper",
    textangle=-90, xanchor="center", yanchor="middle",
    showarrow=False,
    font=dict(size=32),
)
fig.show()


In [84]:
import pandas as pd

# ── Recipe selection (0-based row indices in the Pareto front CSV) ─────────────
EXPERIMENT_RECIPES = [3,27]   # shown as orange diamonds,  e.g. [0, 5, 12]
HIGHLIGHT_RECIPES  = []   # shown as coloured X marks, e.g. [3, 7]

# ── Colour palette for highlight X marks (one colour per recipe) ──────────────
HIGHLIGHT_COLORS = [
    "#E72031", "#108173", "#E9AC11", "#EC720E",
    "#05374B", "#7AF6FA", "#003A91", "#030912",
]

# ─────────────────────────────────────────────────────────────────────────────
pareto_df = pd.read_csv(
    f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/{ITERATION}/pareto_front_predictions.csv"
)
obj_cols = [c for c in pareto_df.columns if c.startswith("obj:")]

num_objectives = len(obj_cols)
all_pairs = [(i, j) for i in range(num_objectives) for j in range(i + 1, num_objectives)]
n_cols = min(3, len(all_pairs))
row1_pairs = all_pairs[:n_cols]
axis_refs = [("x", "y"), ("x2", "y2"), ("x3", "y3")][:n_cols]

for subplot_idx, ((i, j), (xref, yref)) in enumerate(zip(row1_pairs, axis_refs)):
    x_col, y_col = obj_cols[i], obj_cols[j]

    # Orange diamonds
    if EXPERIMENT_RECIPES:
        sel = pareto_df.iloc[EXPERIMENT_RECIPES]
        fig.add_trace(go.Scatter(
            x=sel[x_col], y=sel[y_col],
            mode="markers",
            marker=dict(
                size=18,
                symbol="x",
                color="#FFA500",
                line=dict(width=2, color="black"),
            ),
            name="Experiments",
            legendgroup="experiments",
            xaxis=xref, yaxis=yref,
            showlegend=False
        ))

    # Coloured X marks — one trace per recipe so each gets its own legend entry
    for color_idx, rec_idx in enumerate(HIGHLIGHT_RECIPES):
        row = pareto_df.iloc[rec_idx]
        fig.add_trace(go.Scatter(
            x=[row[x_col]], y=[row[y_col]],
            mode="markers",
            marker=dict(
                size=20,
                symbol="x",
                color=HIGHLIGHT_COLORS[color_idx % len(HIGHLIGHT_COLORS)],
                line=dict(width=3),
            ),
            name=f"Recipe #{rec_idx}",
            legendgroup=f"highlight_{rec_idx}",
            showlegend=False,
            xaxis=xref, yaxis=yref,
        ))
fig.update_layout(showlegend=True, margin=dict(l=100, r=20, t=20, b=100), width=2000, height=600,legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.375, font=dict(size=30)))
fig.show()
fig.write_image(f"{PDF_FOLDER}/robustness_pareto.pdf")
# Extract col-2 subplot (xaxis2/yaxis2) as a standalone figure
_ld = fig.to_plotly_json()["layout"]
robust_fig = go.Figure()
robust_fig.update_layout(template=fig.layout.template)
for _t in fig.data:
    _td = _t.to_plotly_json()
    if (_td.get("xaxis") or "x") == "x2":
        _td = {**_td, "xaxis": "x", "yaxis": "y"}
        robust_fig.add_trace(go.Figure({"data": [_td]}).data[0])
if "xaxis2" in _ld:
    robust_fig.update_layout(xaxis={k: v for k, v in _ld["xaxis2"].items() if k != "domain"})
if "yaxis2" in _ld:
    robust_fig.update_layout(yaxis={k: v for k, v in _ld["yaxis2"].items() if k != "domain"})
for _s in fig.layout.shapes:
    _sd = _s.to_plotly_json()
    if (_sd.get("xref") or "").split(" ")[0] == "x2":
        _x0, _x1 = _sd.get("x0"), _sd.get("x1")
        _y0, _y1 = _sd.get("y0"), _sd.get("y1")
        if _x0 is not None and _x0 == _x1:
            robust_fig.add_vline(x=float(_x0), line=_sd.get("line", {}))
        elif _y0 is not None and _y0 == _y1:
            robust_fig.add_hline(y=float(_y0), line=_sd.get("line", {}))


2026-10-05 16:20:37,931 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:37,932 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:37,932 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpsctv8oi0.
2026-10-05 16:20:37,934 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:37,935 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpjz5rr9fq.
2026-10-05 16:20:37,935 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpjz5rr9fq
2026-10-05 16:20:38,412 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpsctv8oi0/index.html
2026-10-05 16:20:38,976 - kaleido.kaleido - INFO - Getting tab from queue 

In [85]:
from paretodo.dashboard.callbacks.gp_prediction import store_gp_predictions_figures
from paretodo.data_models import InvestigationSpace
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# Load default filter values: first (sorted) value per variable in the meshgrid
_meshgrid_df = pd.read_csv(f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/{ITERATION}/meshgrid_predictions.csv")
_inv_space = InvestigationSpace.from_json(
    Path(f"{LOCAL_TEMP_DIR}/{MLFLOW_RUN}/general/investigation_space.json")
)
_var_names = [v.name for v in _inv_space.variables]
_filter_values = [float(sorted(_meshgrid_df[v].unique())[0]) for v in _var_names]

INVESTIGATION_SPACE_RENAMES = {"temperature": "Temperature [°C]", "ph_value": "pH [-]"}
OBJECTIVES_BOUNDS = {}

gp_figs = store_gp_predictions_figures(
    mlflow_run=MLFLOW_RUN,
    iteration=ITERATION,
    filter_values=_filter_values,
    measurement_renames=MEASUREMENT_RENAMES,
    investigation_space_renames=INVESTIGATION_SPACE_RENAMES,
    objectives_bounds=OBJECTIVES_BOUNDS,
    robustness_confidence_level=None,
    robustness_pareto_point=None,
    alternative_expected_errors=None,
)
x_col, y_col = "temperature", "ph_value"
GP_MATCH = "product_concentration"
for fig_dict in gp_figs:
    title = fig_dict.get("layout", {}).get("title", {}).get("text", "")
    if GP_MATCH not in title:
        continue
    fig = go.Figure(fig_dict)
    fig.data[0].showlegend = False
    fig.data[1].showlegend = True
    fig.data[1].name = "Experiments"
    if EXPERIMENT_RECIPES:
        sel = pareto_df.iloc[EXPERIMENT_RECIPES]
        fig.add_trace(go.Scatter(
            x=sel[x_col], y=sel[y_col],
            mode="markers",
            marker=dict(
                size=18,
                symbol="x",
                color="#FFA500",
                line=dict(width=2, color="black"),
            ),
            name="Selected Recipes",
        ))
    fig.update_layout(width=1050, height=700, margin=dict(l=100, r=20, t=90, b=110), legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.2, font=dict(size=30)), title=dict(text="GP Prediction: Titer with Robustness", font=dict(size=26), x=0.5, xanchor="center"))
    fig.update_xaxes(title_font_size=20, tickfont=dict(size=20))
    fig.update_yaxes(title_font_size=20, tickfont=dict(size=20))
    fig.show()
    fig.write_image(f"{PDF_FOLDER}/robustness_gp.pdf")
    rob_gp_fig = fig


2026-10-05 16:20:39,353 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:39,353 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-10-05 16:20:39,364 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:39,365 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:39,366 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpjpmobbbh.
2026-10-05 16:20:39,367 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:39,367 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp_udgpe23.
2026-10-05 16:20:39,368 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp_udgpe23
2026-10-05 16:20:39,794 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpjpmobbbh/index.html
2026-10-05 16:20:40,381 - kaleido.kaleido - INFO - Getting tab from queue 

In [86]:
import re
import plotly.graph_objects as go
from plotly.subplots import make_subplots

_axis_skip = {"domain", "anchor", "overlaying", "side", "position"}
GP_CA = "coloraxis"

subplot_robust = make_subplots(rows=1, cols=2, horizontal_spacing=0.2,
                               subplot_titles=["GP Visualization", "Pareto Front"])
subplot_robust.update_layout(template=rob_gp_fig.layout.template)

# ── Col 1: GP prediction (rob_gp_fig) ─────────────────────────────────────────
_gp_layout = rob_gp_fig.to_plotly_json()["layout"]

# Set up shared coloraxis from the GP figure (handles both inline and coloraxis sources)
_gp_ca_props = {}
_src_cas = sorted(k for k in _gp_layout if re.match(r"^coloraxis\d*$", k))
if _src_cas:
    _gp_ca_props = {k: v for k, v in _gp_layout[_src_cas[0]].items()
                    if k not in ("cmin", "cmax", "cauto")}
else:
    for _t in rob_gp_fig.data:
        _td0 = _t.to_plotly_json()
        if "z" in _td0 and _td0.get("colorscale"):
            _gp_ca_props = {"colorscale": _td0["colorscale"]}
            break
_gp_ca_props.update({
    "showscale": True,
    "colorbar": {"title": {"text": "Titer [mg/L]", "font": {"size": 25}}},
})
subplot_robust.update_layout({GP_CA: _gp_ca_props})

for _t in rob_gp_fig.data:
    _td = _t.to_plotly_json()
    # Force z-traces onto the shared coloraxis so only one colorbar appears
    if "z" in _td:
        _td = {k: v for k, v in _td.items() if k not in ("showscale", "colorscale", "colorbar")}
        _td["coloraxis"] = GP_CA
    subplot_robust.add_trace(go.Figure({"data": [_td]}).data[0], row=1, col=1)

# Copy axis titles/ranges from rob_gp_fig
for _k, _v in _gp_layout.items():
    if not isinstance(_v, dict):
        continue
    if _k == "xaxis":
        subplot_robust.update_layout({"xaxis": {k: v for k, v in _v.items() if k not in _axis_skip}})
    elif _k == "yaxis":
        subplot_robust.update_layout({"yaxis": {k: v for k, v in _v.items() if k not in _axis_skip}})

# ── Col 2: Pareto front (robust_fig) ──────────────────────────────────────────
_pf_layout = robust_fig.to_plotly_json()["layout"]

for _t in robust_fig.data:
    subplot_robust.add_trace(_t, row=1, col=2)

# Copy axis titles/ranges from robust_fig
for _k, _v in _pf_layout.items():
    if not isinstance(_v, dict):
        continue
    if _k == "xaxis":
        subplot_robust.update_layout({"xaxis2": {k: v for k, v in _v.items() if k not in _axis_skip}})
    elif _k == "yaxis":
        subplot_robust.update_layout({"yaxis2": {k: v for k, v in _v.items() if k not in _axis_skip}})

# Copy shapes (hlines/vlines) from robust_fig
for _s in robust_fig.layout.shapes:
    _sd = _s.to_plotly_json()
    _x0, _x1 = _sd.get("x0"), _sd.get("x1")
    _y0, _y1 = _sd.get("y0"), _sd.get("y1")
    _line = _sd.get("line", {})
    if _x0 is not None and _x0 == _x1:
        subplot_robust.add_vline(x=float(_x0), row=1, col=2, line=_line)
    elif _y0 is not None and _y0 == _y1:
        subplot_robust.add_hline(y=float(_y0), row=1, col=2, line=_line)

# Diagonal gray dashed line on right plot spanning the data range
_xv, _yv = [], []
for _t in subplot_robust.data:
    if (_t.xaxis or "x") == "x2":
        if getattr(_t, "x", None) is not None:
            _xv.extend(v for v in _t.x if isinstance(v, (int, float)))
        if getattr(_t, "y", None) is not None:
            _yv.extend(v for v in _t.y if isinstance(v, (int, float)))
if _xv and _yv:
    _dmin = min(min(_xv), min(_yv))
    _dmax = max(max(_xv), max(_yv))
    subplot_robust.add_shape(
        type="line", x0=_dmin, y0=_dmin, x1=_dmax, y1=_dmax,
        row=1, col=2,
        line=dict(color="gray", width=2, dash="dash"),
    )

subplot_robust.update_layout(
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.61,
                font=dict(size=30), orientation="v", entrywidth=500),
    margin=dict(l=100, r=50, t=100, b=120),
    width=1500, height=700,
)
for annotation in subplot_robust.layout.annotations:
    annotation.font.size = 30
subplot_robust.show()
subplot_robust.write_image(f"{PDF_FOLDER}/pfgs_robustness.pdf")

2026-10-05 16:20:40,689 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-10-05 16:20:40,690 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-10-05 16:20:40,706 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-10-05 16:20:40,707 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-10-05 16:20:40,708 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp1wqnxp3r.
2026-10-05 16:20:40,708 - choreographer.browser_async - INFO - Opening browser.
2026-10-05 16:20:40,709 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp3iljqnq1.
2026-10-05 16:20:40,709 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp3iljqnq1
2026-10-05 16:20:41,146 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp1wqnxp3r/index.html
2026-10-05 16:20:41,763 - kaleido.kaleido - INFO - Getting tab from queue 